# Pilot study: is the calibrated margin worth it? (Step 1: why `tire_stiff` failed · Step 2: moving obstacles & obstacle-position error)
**Purpose.** A small, *pre-specified* experiment that decides whether the paper should be a **method paper** (path A) or an **evaluation/benchmark paper** (path B). Nothing here is tuned to make the proposed controller win.

**What the notebook does**
1. Re-creates the plants (home single-track / independent 29-state multi-body), the neural-residual ensemble, the NMPC, the friction estimator and the ego-model conformal tube from the main notebook (Sections 1-8, unchanged code).
2. **Step 1 – diagnosis:** controlled interventions on the `tire_stiff` scenario (null tube, CBF-form off, `q_cap`, no ego-tube margin on the road edges, ...) on *diag* seeds → automatic verdict → a **fix chosen on diag seeds and frozen**.
3. **Step 2 – moving obstacles and obstacle-position error:** 6 new scenarios (constant velocity control, swerve, stop-and-go, heavy-tailed measurement noise, combinations, low grip), a sliding-window tracker, an **obstacle-side conformal margin** (online ACI with delayed feedback, calibrated offline on nominal obstacle behaviour), and **strong baselines**: adaptive MPC, a **tuned velocity-uncertainty inflation** (robust-MPC-like) and a **predictive CBF safety filter** (two variants), plus static split-CP and an oracle reference.
4. Paired exact tests (Holm), bootstrap CIs, Wilson CIs, pre-specified hypotheses **HA-HE** and an **automatic decision** (path A / A' / B).
5. Exports `PILOT_REPORT.md` — **send this file back for evaluation**.

**How to run (Colab, CPU):** set `PLANT_CHOICE` / `PILOT_CHOICE` in the next cell and *Run all*. `smoke` ≈ wiring check · `quick` (8 test seeds) · `pilot30` (30 test seeds). Rough time estimates (not measured for this exact notebook; based on ~5-9 s per episode on 2 CPUs): quick ≈ 45-60 min, pilot30 ≈ 2-3 h. Everything is **resumable** — re-run the cell after a disconnect (set `USE_DRIVE = True` to keep results across runtime resets).

**Honest status of this notebook.** It was assembled from code that was run and validated earlier (the main notebook), but **the new parts (moving-obstacle world, tracker, obstacle tube, CBF filter, Step 1/2 cells) were not executed before delivery** — only syntax/undefined-name checks were done. The first cells therefore include a **regression test** (new simulator vs. validated simulator) and a **scenario-difficulty gate** (stops the run if the new scenarios are degenerate). If something crashes, send me the traceback.


In [ ]:
# ---- choices (edit here) ----
PLANT_CHOICE = "mb"        # "mb" = independent 29-state multi-body plant (recommended; Step 1 failed on it) | "home" = own single-track plant
PILOT_CHOICE = "quick"     # "smoke" (wiring check, few minutes) | "quick" (8 test seeds) | "pilot30" (30 test seeds)
NN_MODE      = "fast"      # size of the offline data / network training (the main notebook's modes: smoke | fast | standard)
DELAY_CHOICE = 1           # computational delay in control steps
USE_DRIVE    = False       # True: mount Google Drive and store the results there (survives runtime resets)
# -----------------------------
import os, sys, subprocess, importlib
for pkg, mod in [("casadi", "casadi"), ("commonroad-vehicle-models", "vehiclemodels")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
os.environ.setdefault("CA_MODE", NN_MODE); os.environ.setdefault("CA_PLANT", PLANT_CHOICE); os.environ.setdefault("CA_PILOT", PILOT_CHOICE); os.environ.setdefault("CA_DELAY", str(DELAY_CHOICE))
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); os.environ["CA_OUTROOT"] = "/content/drive/MyDrive/ca_nmpc_pilot"; os.makedirs(os.environ["CA_OUTROOT"], exist_ok=True)
print({k: os.environ[k] for k in ("CA_MODE", "CA_PLANT", "CA_PILOT", "CA_DELAY")})

In [ ]:
import os, sys, time, math, json, subprocess, warnings, random, itertools, pickle
from dataclasses import dataclass
from collections import deque
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    import casadi as ca
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "casadi"], check=True)
    import casadi as ca
import torch
import torch.nn as nn

# ------------------------------------------------------------------ CONFIG
MODE = os.environ.get("CA_MODE", "fast")      # "smoke" | "fast" | "standard" | "full"
PLANT = os.environ.get("CA_PLANT", "home")    # "home": own single-track plant | "mb": independent 29-state multi-body plant (CommonRoad, BMW 320i parameters)
DELAY = int(os.environ.get("CA_DELAY", "1"))  # computational delay in control steps (0 = instantaneous solve, 1 = command computed during step k is applied at step k+1)
SC_ALL = ["nominal", "wet_heavy", "ice", "mu_switch", "gust_wet", "mass_cog", "tire_stiff", "sensor_bias", "steer_fault", "popout_wet"]
FAM_CONTROL = ["nominal"]                                                     # in-distribution control scenario
FAM_FRIC = ["wet_heavy", "ice", "mu_switch"]                                  # grip shifts (a 1-parameter friction scale can explain most of them)
FAM_OTHER = ["gust_wet", "mass_cog", "tire_stiff", "sensor_bias", "steer_fault", "popout_wet"]   # shifts / conditions NOT representable by the friction scale
DEV_SEEDS = [0, 1, 2, 3]                       # development / tuning seeds (tube tuning, sensitivity sweeps, scenario severity) -- never used for reporting
CFGS = {   # evaluation (test) seeds start at 1000 and are never used for tuning
    "smoke":    dict(seeds=[1000],                  scenarios=["nominal", "ice"], T_sim=5.0, n_train=40,  n_val=10, n_cal=20, epochs=40),
    "fast":     dict(seeds=list(range(1000, 1003)), scenarios=SC_ALL,             T_sim=8.5, n_train=120, n_val=20, n_cal=40, epochs=120),
    "standard": dict(seeds=list(range(1000, 1010)), scenarios=SC_ALL,             T_sim=8.5, n_train=300, n_val=40, n_cal=80, epochs=200),
    "full":     dict(seeds=list(range(1000, 1030)), scenarios=SC_ALL,             T_sim=8.5, n_train=300, n_val=40, n_cal=80, epochs=200),
}
CFG = dict(CFGS[MODE])
if os.environ.get("CA_NSEEDS"): CFG["seeds"] = list(range(1000, 1000 + int(os.environ["CA_NSEEDS"])))     # optional overrides (e.g. to shorten a run)
if os.environ.get("CA_SCEN"): CFG["scenarios"] = os.environ["CA_SCEN"].split(",")
OUT = f"results_{PLANT}" if MODE != "smoke" else f"results_{PLANT}_smoke"; os.makedirs(OUT, exist_ok=True)
X_END = 100.0       # an episode ends when the vehicle passes x = 100 m (both obstacles are behind it)

DT = 0.1            # control / model step [s]
N = 25              # MPC horizon steps (2.5 s)
NX, NU = 8, 2       # states [x y psi vx vy r delta ax], inputs [delta_cmd, ax_cmd]
DYN = [3, 4, 5, 6, 7]   # states on which the residual acts
ND = len(DYN)
K_ENS, HID = 5, 32
SEED0 = 12345

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(SEED0)
torch.set_num_threads(max(1, (os.cpu_count() or 2)))
print("MODE:", MODE, "| PLANT:", PLANT, "| DELAY:", DELAY, "|", CFG)
print("casadi", ca.__version__, "| torch", torch.__version__, "| cpu cores", os.cpu_count())

## 1. Vehicle models
*Plant (ground truth):* dynamic single-track model with **Pacejka magic-formula tyres whose peak scales with friction `mu`**, a
**friction ellipse** (braking/accelerating reduces lateral capacity), **steering-rate limit**, first-order actuator lags,
aerodynamic drag, rolling resistance, and the road-friction-limited acceleration.

*Nominal model (what the controller knows):* a **saturating (tanh) tyre model that assumes a dry road (`mu0 = 0.9`)**, with a *friction-scale parameter `xi`* (`xi = 1` dry) that the controller can identify online,
but with different stiffness/inertia/actuator constants, no friction ellipse and no steering-rate limit. The controller never sees the true `mu`.
(Using a saturating nominal tyre makes the baselines *strong*: the only first-order unknown is grip, as in practice.)

In [ ]:
@dataclass
class Veh:
    lf: float = 1.2; lr: float = 1.5
    m0: float = 1500.0; Iz0: float = 2500.0
    g: float = 9.81
    # Pacejka (true)
    B: float = 9.0; C: float = 1.3; E: float = -0.5
    # true actuators / resistances
    tau_d: float = 0.12; tau_a: float = 0.25; rate_max: float = 0.8
    cd: float = 0.42; crr: float = 0.015
    # nominal model parameters (mismatched on purpose)
    m_nom: float = 1500.0; Iz_nom: float = 2500.0
    Cf_nom: float = 80000.0; Cr_nom: float = 72000.0; mu0_nom: float = 0.9
    tau_d_nom: float = 0.20; tau_a_nom: float = 0.35
    cd_nom: float = 0.30; crr_nom: float = 0.010
    # limits
    d_max: float = 0.35; ax_min: float = -6.0; ax_max: float = 3.0
if PLANT == "mb":
    try:
        from vehiclemodels.parameters_vehicle2 import parameters_vehicle2
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "commonroad-vehicle-models"], check=True)
        from vehiclemodels.parameters_vehicle2 import parameters_vehicle2
    from vehiclemodels.init_mb import init_mb
    from vehiclemodels.vehicle_dynamics_mb import vehicle_dynamics_mb
    _P0 = parameters_vehicle2()          # BMW-320i-like parameter set shipped with CommonRoad vehicle models
    # the controller knows the vehicle's geometry/mass/inertia but NOT its tyre law: nominal tyre = saturating tanh with a *mismatched* stiffness
    V = Veh(lf=_P0.a, lr=_P0.b, m0=_P0.m, Iz0=_P0.I_z, m_nom=_P0.m, Iz_nom=_P0.I_z, Cf_nom=1.0e5, Cr_nom=9.0e4,
            B=abs(_P0.tire.p_ky1) / (_P0.tire.p_cy1 * _P0.tire.p_dy1), C=_P0.tire.p_cy1, E=_P0.tire.p_ey1)     # (B, C, E) only used by the single-track *oracle*
    ORACLE_MU_GAIN = _P0.tire.p_dy1 / 0.9
else:
    V = Veh(); ORACLE_MU_GAIN = 1.0
L_WB = V.lf + V.lr

def _smooth_vx(vx):
    return 0.5 * (vx + 1.0 + ca.sqrt((vx - 1.0) ** 2 + 0.1))

def _kin(s):
    x, y, psi, vx, vy, r = s[0], s[1], s[2], s[3], s[4], s[5]
    return ca.vertcat(vx * ca.cos(psi) - vy * ca.sin(psi), vx * ca.sin(psi) + vy * ca.cos(psi), r)

def _pac(alpha, D, cs=1.0):
    Ba = V.B * cs * alpha
    return D * ca.sin(V.C * ca.atan(Ba - V.E * (Ba - ca.atan(Ba))))

def f_plant_ct(s, u, th):
    """true continuous-time model. th = [mu, mass, cornering-stiffness scale cs, CoG shift (positive = CoG moves rearwards)]"""
    mu, m, cs, cog = th[0], th[1], th[2], th[3]
    Iz = V.Iz0 * m / V.m0
    lf, lr = V.lf + cog, V.lr - cog
    vx, vy, r, dl, ax = s[3], s[4], s[5], s[6], s[7]
    vxs = _smooth_vx(vx)
    Fzf = m * V.g * lr / L_WB; Fzr = m * V.g * lf / L_WB
    axlim = 0.95 * mu * V.g
    ax_eff = axlim * ca.tanh(ax / axlim)
    rho = ax_eff / (mu * V.g)
    ell = ca.sqrt(1.0 - rho ** 2)                      # friction ellipse factor
    af = dl - ca.atan((vy + lf * r) / vxs)
    ar = -ca.atan((vy - lr * r) / vxs)
    Fyf = ell * _pac(af, mu * Fzf, cs); Fyr = ell * _pac(ar, mu * Fzr, cs)
    dvx = ax_eff + vy * r - V.cd * vx ** 2 / m - V.crr * V.g
    dvy = (Fyf * ca.cos(dl) + Fyr) / m - vx * r
    dr = (lf * Fyf * ca.cos(dl) - lr * Fyr) / Iz
    ddl = V.rate_max * ca.tanh((u[0] - dl) / (V.tau_d * V.rate_max))
    dax = (u[1] - ax) / V.tau_a
    return ca.vertcat(_kin(s), dvx, dvy, dr, ddl, dax)

def f_nom_ct(s, u, th):
    """nominal (controller) model: SATURATING (tanh) tyres assuming dry road mu0=0.9, wrong stiffness/inertia/lag constants,
    no friction ellipse, no steering-rate limit. th[0] = xi is the friction scale (xi=1: dry). The controller never sees the true mu."""
    xi = th[0]
    m, Iz = V.m_nom, V.Iz_nom
    vx, vy, r, dl, ax = s[3], s[4], s[5], s[6], s[7]
    vxs = _smooth_vx(vx)
    Fzf = m * V.g * V.lr / L_WB; Fzr = m * V.g * V.lf / L_WB
    D0f, D0r = V.mu0_nom * Fzf, V.mu0_nom * Fzr
    af = dl - ca.atan((vy + V.lf * r) / vxs); ar = -ca.atan((vy - V.lr * r) / vxs)
    Fyf = xi * D0f * ca.tanh(V.Cf_nom * af / D0f); Fyr = xi * D0r * ca.tanh(V.Cr_nom * ar / D0r)
    axlim = xi * 0.95 * V.mu0_nom * V.g
    ax_eff = axlim * ca.tanh(ax / axlim)
    dvx = ax_eff + vy * r - V.cd_nom * vx ** 2 / m - V.crr_nom * V.g
    dvy = (Fyf * ca.cos(dl) + Fyr) / m - vx * r
    dr = (V.lf * Fyf * ca.cos(dl) - V.lr * Fyr) / Iz
    ddl = (u[0] - dl) / V.tau_d_nom
    dax = (u[1] - ax) / V.tau_a_nom
    return ca.vertcat(_kin(s), dvx, dvy, dr, ddl, dax)

def make_step(f_ct, n_sub, name, n_th=2):
    s = ca.SX.sym("s", NX); u = ca.SX.sym("u", NU); th = ca.SX.sym("th", n_th)
    h = DT / n_sub; x = s
    for _ in range(n_sub):
        k1 = f_ct(x, u, th); k2 = f_ct(x + h / 2 * k1, u, th)
        k3 = f_ct(x + h / 2 * k2, u, th); k4 = f_ct(x + h * k3, u, th)
        x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return ca.Function(name, [s, u, th], [x])

F_plant = make_step(f_plant_ct, 4, "F_plant", 4)        # simulation truth of the "home" plant (4 RK4 sub-steps)
F_true_mpc = make_step(f_plant_ct, 2, "F_true_mpc", 4)  # oracle MPC model (single-track physics with the true mu/mass/stiffness/CoG, MPC discretisation)
F_nom = make_step(f_nom_ct, 2, "F_nom")              # nominal MPC model
_map_cache = {}
def fmap(F, n):
    key = (F.name(), n)
    if key not in _map_cache:
        _map_cache[key] = F.map(n)
    return _map_cache[key]

def np_step(F, S, U, TH):
    """batched numpy step. S:(B,8) U:(B,2) TH:(B,2) -> (B,8)"""
    B = S.shape[0]
    return np.array(fmap(F, B)(S.T, U.T, TH.T)).T

TH_NOM_TRAIN = (1.0, 0.0)      # nominal model parameter vector: [xi (friction scale), unused]
OFFLINE_NOISE = 0.2
NOISE_PROC = np.array([0, 0, 0, 0, 0.02, 0.002, 0, 0.0])         # added after each plant step
NOISE_MEAS = np.array([0.03, 0.03, 0.002, 0.05, 0.03, 0.003, 0.001, 0.05])
print("models compiled")

### 1.1 Plants (ground truth the controllers are evaluated on)
* `home`: the single-track plant above (own implementation, Pacejka tyres, friction ellipse, rate limit, lags).
* `mb`: an **independent 29-state multi-body model** (CommonRoad *vehicle-models*, DOT multi-body: roll/pitch/heave, axle & wheel dynamics, load transfer, combined-slip Pacejka tyres, BMW-320i-like parameters).
  Its model family is different from the controller's nominal model, so the model mismatch is *structural* and not designed by us. Friction is changed by scaling the Pacejka peak factors (`p_dy1`, `p_dx1`), tyre stiffness by `p_ky1`,
  mass by the sprung mass (and inertias), CoG by moving the axle distances. Both plants expose the same interface: `reset(v0)` and `step(u, t, proc_noise, dvy)`.
**Friction convention.** `mu = 0.9` is "dry/in-distribution"; for `mb` the native tyre has a peak of about 1.05, so the friction *scale* is `mu / 0.9`.

In [ ]:
def mu_at(sc, t):
    return sc["mu2"] if ("mu2" in sc and t >= sc["t_switch"]) else sc["mu"]

def apply_fault(sc, u, t):
    """actuator fault (unmodelled): after fault_t the steering actuator loses gain and gets an offset"""
    if "fault_t" in sc and t >= sc["fault_t"]:
        return np.array([sc.get("steer_gain", 1.0) * u[0] + sc.get("steer_off", 0.0), u[1]])
    return u

class HomePlant:
    def __init__(self, sc): self.sc = sc
    def reset(self, v0): self.s = np.array([0, 0, 0, v0, 0, 0, 0, 0.0]); return self.s.copy()
    def step(self, u, t, pn, dvy=0.0):
        sc = self.sc; th = np.array([mu_at(sc, t), sc["m"], sc.get("cs", 1.0), sc.get("cog", 0.0)]); u = apply_fault(sc, u, t)
        self.s = np.array(F_plant(self.s, u, th)).ravel() + pn
        self.s[4] += dvy
        return self.s.copy()

class MBPlant:
    SUB = 10
    def __init__(self, sc): self.sc = sc; self._cache = {}
    def _params(self, mu):
        key = (round(mu, 4), self.sc.get("cs", 1.0), self.sc["m"], self.sc.get("cog", 0.0))
        if key not in self._cache:
            p = parameters_vehicle2()
            sc_mu = mu / 0.9
            p.tire.p_dy1 *= sc_mu; p.tire.p_dx1 *= sc_mu; p.tire.p_ky1 *= self.sc.get("cs", 1.0)
            ms = self.sc["m"] / 1500.0
            if abs(ms - 1.0) > 1e-9:
                d = p.m_s * (ms - 1.0); p.m_s += d; p.m += d
                for a_ in ("I_Phi_s", "I_y_s", "I_z"): setattr(p, a_, getattr(p, a_) * ms)
            p.a += self.sc.get("cog", 0.0); p.b -= self.sc.get("cog", 0.0)
            self._cache[key] = p
        return self._cache[key]
    def reset(self, v0):
        p = self._params(mu_at(self.sc, 0.0)); self.x = np.array(init_mb([0, 0, 0, v0, 0, 0, 0], p), dtype=float); self.a = 0.0
        return self._s8()
    def _s8(self): x = self.x; return np.array([x[0], x[1], x[4], x[3], x[10], x[5], x[2], self.a])
    def _f(self, x, u, p): return np.array(vehicle_dynamics_mb(x, u, p))
    def step(self, u, t, pn, dvy=0.0):
        p = self._params(mu_at(self.sc, t)); h = DT / self.SUB; x = self.x; u = apply_fault(self.sc, u, t)
        for _ in range(self.SUB):
            ddl = V.rate_max * np.tanh((u[0] - x[2]) / (V.tau_d * V.rate_max))
            self.a += h * (u[1] - self.a) / V.tau_a
            uu = [ddl, self.a]
            k1 = self._f(x, uu, p); k2 = self._f(x + h / 2 * k1, uu, p); k3 = self._f(x + h / 2 * k2, uu, p); k4 = self._f(x + h * k3, uu, p)
            x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        x[10] += pn[4] + dvy; x[5] += pn[5]
        self.x = x
        return self._s8()

def make_plant(sc): return MBPlant(sc) if PLANT == "mb" else HomePlant(sc)

## 2. Data: excitation trajectories on the **training domain only** (high friction), generated by the selected plant
Training domain: `mu ~ U[0.8, 1.0]`, `m ~ U[1450, 1550]` (nominal cornering stiffness and CoG). The test scenarios (wet, ice, abrupt friction switch, mass/CoG, softer tyres, ...) are **out of distribution** by design.
This data plays the role of logged driving data: in a real deployment it would come from the vehicle itself.

In [ ]:
def gen_traj(rng, mu, m, T_steps=100, noise_scale=1.0):
    """open-loop excitation trajectory on the plant; returns measured states S (T+1,8), inputs U (T,2), truth Str"""
    plant = make_plant({"mu": mu, "m": m})
    v_t = rng.uniform(8, 16)
    s = plant.reset(v_t)
    nsin = 3
    A = rng.uniform(0.0, 0.07, nsin) * (12.0 / v_t); fr = rng.uniform(0.1, 0.7, nsin); ph = rng.uniform(0, 2 * np.pi, nsin)
    step_t = rng.integers(10, 60); step_a = rng.uniform(-0.06, 0.06) * (12.0 / v_t) * (rng.random() < 0.6)
    Str = [s.copy()]; U = []
    for k in range(T_steps):
        t = k * DT
        dcmd = float((A * np.sin(2 * np.pi * fr * t + ph)).sum() + (step_a if k >= step_t else 0.0))
        if rng.random() < 0.15: dcmd *= rng.uniform(1.0, 1.8)
        acmd = float(np.clip(0.8 * (v_t - s[3]) + 0.8 * np.sin(2 * np.pi * rng.uniform(0.1, 0.4) * t), V.ax_min + 2, V.ax_max))
        dcmd = float(np.clip(dcmd, -V.d_max, V.d_max)); u = np.array([dcmd, acmd])
        s = plant.step(u, t, NOISE_PROC * rng.standard_normal(NX))
        U.append(u); Str.append(s.copy())
        if abs(s[4]) > 3.5 or s[3] < 3.0 or abs(s[2]) > 1.0 or not np.all(np.isfinite(s)): break
    Str = np.array(Str); U = np.array(U)
    S = Str + noise_scale * NOISE_MEAS * rng.standard_normal(Str.shape)
    return S, U, Str

def make_dataset(rng, n_traj, mu_rng=(0.8, 1.0), m_rng=(1450, 1550), T_steps=100, noise_scale=1.0):
    Z, Y = [], []
    trajs = []
    for _ in range(n_traj):
        mu = rng.uniform(*mu_rng); m = rng.uniform(*m_rng)
        S, U, _ = gen_traj(rng, mu, m, T_steps, noise_scale)
        if len(U) < 5: continue
        trajs.append((S, U, mu, m))
        TH = np.tile(TH_NOM_TRAIN, (len(U), 1))
        pred = np_step(F_nom, S[:-1], U, TH)
        Y.append(S[1:, DYN] - pred[:, DYN])
        Z.append(np.hstack([S[:-1, DYN], U]))
    return np.vstack(Z), np.vstack(Y), trajs

rng_data = np.random.default_rng(SEED0)
t0 = time.time()
# offline training/validation data: lab-grade (fused/filtered) measurements -> 0.2x of the online sensor noise
Ztr, Ytr, traj_tr = make_dataset(rng_data, CFG["n_train"], noise_scale=OFFLINE_NOISE)
Zva, Yva, traj_va = make_dataset(rng_data, CFG["n_val"], noise_scale=OFFLINE_NOISE)
# calibration data: in-distribution but with the FULL online sensor noise (it calibrates the deployed system)
Zcal, Ycal, traj_cal = make_dataset(rng_data, CFG["n_cal"], noise_scale=1.0)
Zood, Yood, traj_ood = make_dataset(rng_data, 20, mu_rng=(0.3, 0.45), m_rng=(1450, 1550), noise_scale=OFFLINE_NOISE)  # OOD diagnostic only
print(f"data: train {Ztr.shape}, val {Zva.shape}, cal {Zcal.shape}, ood {Zood.shape} in {time.time()-t0:.1f}s")

## 3. Deep-ensemble residual (PyTorch) and its CasADi-embeddable form
`K=5` MLPs `7 -> 32 -> 32 -> 5` (tanh, so the NMPC stays smooth) are trained with bootstrap resampling. Their **mean is distilled into one student
`7 -> 48 -> 48 -> 5`** that runs inside the NMPC (5x cheaper), while the members are kept for the uncalibrated *ensemble-spread* baseline.
The student is **linear in its last layer**: `Delta_hat(z) = Theta * Phi(z)`, `Phi = [h2 | 1]` (h2 = 48 hidden features). `Phi` and `Theta` are **frozen after offline training**;
online adaptation is done in a *physics-structured, 1-parameter* way instead (friction scale `xi`, Section 7) because adapting a high-dimensional last layer from noisy data on a few seconds of driving proved unstable (collinear features) in our development runs.

In [ ]:
class MLP(nn.Module):
    def __init__(self, din, h, dout):
        super().__init__()
        self.l1 = nn.Linear(din, h); self.l2 = nn.Linear(h, h); self.l3 = nn.Linear(h, dout)
    def forward(self, x, feats=False):
        h2 = torch.tanh(self.l2(torch.tanh(self.l1(x))))
        return (self.l3(h2), h2) if feats else self.l3(h2)

z_mu, z_sd = Ztr.mean(0), Ztr.std(0) + 1e-6
y_sd = Ytr.std(0) + 1e-9
def zn(Z): return (Z - z_mu) / z_sd

def train_ensemble(Ztr, Ytr, Zva, Yva, epochs, seed=0):
    X = torch.tensor(zn(Ztr), dtype=torch.float32); Y = torch.tensor(Ytr / y_sd, dtype=torch.float32)
    Xv = torch.tensor(zn(Zva), dtype=torch.float32); Yv = torch.tensor(Yva / y_sd, dtype=torch.float32)
    members, curves = [], []
    for k in range(K_ENS):
        torch.manual_seed(seed * 100 + k)
        net = MLP(Ztr.shape[1], HID, ND); opt = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        idx_boot = torch.randint(0, len(X), (len(X),))            # bootstrap
        Xb, Yb = X[idx_boot], Y[idx_boot]
        best, best_state, curve = 1e9, None, []
        for ep in range(epochs):
            perm = torch.randperm(len(Xb))
            for i in range(0, len(Xb), 256):
                b = perm[i:i + 256]
                loss = ((net(Xb[b]) - Yb[b]) ** 2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            with torch.no_grad(): vl = ((net(Xv) - Yv) ** 2).mean().item()
            curve.append(vl)
            if vl < best: best, best_state = vl, {k_: v.clone() for k_, v in net.state_dict().items()}
        net.load_state_dict(best_state); members.append(net); curves.append(curve)
    return members, curves

t0 = time.time()
members, curves = train_ensemble(Ztr, Ytr, Zva, Yva, CFG["epochs"])
print(f"ensemble trained in {time.time()-t0:.1f}s | best val MSE (normalised): {[round(min(c),4) for c in curves]}")

def extract(nets):
    g = lambda t: t.detach().numpy().astype(float)
    return ([g(m.l1.weight) for m in nets], [g(m.l1.bias) for m in nets], [g(m.l2.weight) for m in nets],
            [g(m.l2.bias) for m in nets], [g(m.l3.weight) for m in nets], [g(m.l3.bias) for m in nets])
MW1, Mb1, MW2, Mb2, MW3, Mb3 = extract(members)           # ensemble members (used for the ensemble-spread baseline)

# ---- distil the ensemble MEAN into one student network: 5x cheaper inside the NMPC, and a single feature map
HID_S = 48
def distill(members, Ztr, epochs, seed=0):
    torch.manual_seed(seed + 777)
    X0 = zn(Ztr); rngd = np.random.default_rng(seed)
    Xa = np.vstack([X0, X0 + 0.10 * rngd.standard_normal(X0.shape), X0 + 0.25 * rngd.standard_normal(X0.shape)])
    Xt = torch.tensor(Xa, dtype=torch.float32)
    with torch.no_grad(): Yt = torch.stack([m(Xt) for m in members]).mean(0)
    net = MLP(Xa.shape[1], HID_S, ND); opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 256):
            b = perm[i:i + 256]; loss = ((net(Xt[b]) - Yt[b]) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    with torch.no_grad(): fid = float(((net(Xt) - Yt) ** 2).mean().sqrt())
    return net, fid
student, fid = distill(members, Ztr, CFG["epochs"])
W1, b1, W2, b2, W3, b3 = [v[0] for v in extract([student])]
print(f"student distilled from ensemble mean (RMSE fidelity {fid:.4f} in normalised units)")
FDIM = HID_S + 1                                           # Phi = [h2 (48) | 1]
THETA0 = np.hstack([W3, b3[:, None]])                      # (ND, FDIM)  -- frozen after offline training

def phi_np(Z):
    """Z raw (B,7) -> Phi (B,FDIM) = [h2, 1]"""
    Zn = zn(Z)
    h1 = np.tanh(Zn @ W1.T + b1); h2 = np.tanh(h1 @ W2.T + b2)
    return np.hstack([h2, np.ones((Z.shape[0], 1))])

def member_np(Zn, k):
    h1 = np.tanh(Zn @ MW1[k].T + Mb1[k]); h2 = np.tanh(h1 @ MW2[k].T + Mb2[k]); return h2 @ MW3[k].T + Mb3[k]

def resid_np(Z, Theta):
    return (phi_np(Z) @ Theta.T) * y_sd

def phi_ca(z):
    """CasADi version of Phi for a (7,1) raw input (SX or MX)"""
    zz = (z - ca.DM(z_mu)) / ca.DM(z_sd)
    h1 = ca.tanh(ca.mtimes(ca.DM(W1), zz) + ca.DM(b1))
    h2 = ca.tanh(ca.mtimes(ca.DM(W2), h1) + ca.DM(b2))
    return ca.vertcat(h2, 1.0)

def rmse(a): return float(np.sqrt((a ** 2).mean()))
def eval_one_step(Z, Y, Theta=THETA0):
    return rmse(Y / y_sd), rmse((Y - resid_np(Z, Theta)) / y_sd)
for name, (Z, Y) in {"val (in-dist)": (Zva, Yva), "OOD (mu 0.3-0.45)": (Zood, Yood)}.items():
    a, b = eval_one_step(Z, Y)
    print(f"1-step normalised RMSE | {name:20s} nominal {a:.3f} -> +NN residual {b:.3f}")

## 4. Unit tests (numpy <-> CasADi consistency) — cheap insurance for reproducibility

In [ ]:
def nn_step_ca_fn():
    s = ca.SX.sym("s", NX); u = ca.SX.sym("u", NU); th = ca.SX.sym("th", 2); Th = ca.SX.sym("Th", ND, FDIM)
    sn = F_nom(s, u, th)
    z = ca.vertcat(s[3:8], u)
    d = ca.mtimes(Th, phi_ca(z)) * ca.DM(y_sd)
    return ca.Function("F_nn", [s, u, th, Th], [sn + ca.vertcat(ca.DM.zeros(3, 1), d)])
F_nn = nn_step_ca_fn()

rng_t = np.random.default_rng(1)
S, U, _ = gen_traj(rng_t, 0.9, 1500.0, 30)
s0, u0 = S[3], U[3]
a = np.array(F_nn(s0, u0, [1.0, 0.0], THETA0)).ravel()
b = np_step(F_nom, s0[None], u0[None], np.array([[1.0, 0.0]]))[0] + np.concatenate([np.zeros(3), resid_np(np.hstack([s0[DYN], u0])[None], THETA0)[0]])
assert np.allclose(a, b, atol=1e-8), (a - b)
# plant energy/sanity: straight line, zero input -> only drag decelerates
sx = np.array([0, 0, 0, 15, 0, 0, 0, 0.0]); sx2 = np.array(F_plant(sx, [0, 0], [0.9, V.m0, 1.0, 0.0])).ravel()
assert abs(sx2[1]) < 1e-9 and abs(sx2[4]) < 1e-9 and sx2[3] < 15
# steady-state turn is left-turning for positive steering
sy = np.array([0, 0, 0, 12, 0, 0, 0.05, 0.0])
for _ in range(30): sy = np.array(F_plant(sy, [0.05, 0.0], [0.9, V.m0, 1.0, 0.0])).ravel()
assert sy[5] > 0.05 and sy[4] != 0
# the active plant (home or multi-body) must also behave sanely: straight line, then a left turn
_pl = make_plant({"mu": 0.9, "m": 1500.0}); _s = _pl.reset(13.0)
for _ in range(10): _s = _pl.step(np.zeros(2), 0.0, np.zeros(NX))
assert abs(_s[1]) < 0.05 and abs(_s[4]) < 0.2 and _s[3] > 12.0, _s
for _ in range(15): _s = _pl.step(np.array([0.04, 0.0]), 0.0, np.zeros(NX))
assert _s[5] > 0.05 and np.all(np.isfinite(_s)), _s
print("unit tests passed ✔  (numpy NN == CasADi NN; plant sanity for plant =", PLANT, ")")

## 5. Scenarios (the distribution-shift benchmark)
A 7.2 m road with a gentle S-shaped lane centre (so lateral dynamics are excited from the start and the shift becomes *observable*), two static obstacles that force an evasive left-then-right manoeuvre (**limited perception range: 32 m**, so it must be planned ~2.5 s ahead). Three families (control / grip / other):

| family | scenario | what changes | why it is in the benchmark |
|---|---|---|---|
| control | `nominal` | mu 0.90, 1500 kg | in-distribution sanity |
| grip | `wet_heavy` | mu 0.55, +20 % mass | parametric OOD shift |
| grip | `ice` | mu 0.35 | strong shift (handling limit ~3.4 m/s²) |
| grip | `mu_switch` | mu 0.90 → 0.40 at t = 1.5 s | abrupt non-stationary shift while the evasive manoeuvre starts |
| **other** | `gust_wet` | mu 0.60 + **unmodelled 3.5 m/s² lateral gust** (1.6 s) | a disturbance, not a grip change: no friction scale can explain it |
| **other** | `mass_cog` | mu 0.50, **+80 % mass, CoG 0.6 m rearwards** | changes the yaw dynamics |
| **other** | `tire_stiff` | mu 0.50, **cornering stiffness −75 %** | the force *slope* changes, not only the peak |
| **other** | `sensor_bias` | mu 0.60, **v_y bias 0.2 m/s, gyro bias 0.08 rad/s, steering-sensor bias 0.02 rad** | systematic measurement error |
| **other** | `steer_fault` | mu 0.70, **steering actuator fault at t = 3 s (gain 0.6, +0.03 rad offset)** | unmodelled actuator degradation |
| **other** | `popout_wet` | mu 0.55, **perception range 12 m** | late detection: little time to react |

*Design rule (decided before looking at the test seeds):* the severity of the "other" scenarios was set on the **development seeds** so that the strongest classical baseline (`Adaptive-MPC`) is neither at 0 % nor at 100 % — **never tuned for our method**.
A **narrow-corridor** scenario was tried and dropped (the obstacle geometry made its feasibility ill-defined even for the oracle). Perception errors of the obstacle position and moving obstacles are **not** covered (model-side scores cannot see them): see the limitations section.
Per-seed randomisation: obstacle positions, initial speed, sensor noise and process disturbances. **All controllers share the same random streams per seed (paired design).**

In [ ]:
R_O, R_V = 0.8, 1.0
R_BUF = 0.15         # inter-sample buffer: constraints are enforced at the 0.1 s grid only (1.2 m apart), so inflate the obstacle by 0.15 m
Y_LO, Y_HI = -2.6, 4.6       # 7.2 m wide road (the left part is the opposite lane / shoulder)
PATH_A, PATH_L = 1.0, 55.0   # lane-centre path: gentle S-curve  y = A sin(2 pi x / L)  (continuous lateral excitation, ~2 m/s^2 peak at 13 m/s)
_KX = 2 * np.pi / PATH_L
def path_y(x): return PATH_A * np.sin(_KX * x)
V_REF = 13.0
R_SENSE = 32.0       # limited perception range [m]: obstacles are unknown to the controller until they are within R_SENSE
M_OBS = 2
SCEN = {
    "nominal":     dict(mu=0.90, m=1500.0),
    "wet_heavy":   dict(mu=0.55, m=1800.0),
    "ice":         dict(mu=0.35, m=1500.0),
    "mu_switch":   dict(mu=0.90, m=1500.0, mu2=0.40, t_switch=1.5),
    "gust_wet":    dict(mu=0.60, m=1500.0, gust_t0=3.0, gust_dur=1.6, gust_a=-3.5),   # UNMODELLED lateral gust of 3.5 m/s^2 for 1.6 s (towards the obstacle side)
    "mass_cog":    dict(mu=0.50, m=2700.0, cog=0.60),                                   # +80 % mass, CoG 0.6 m rearwards, low grip
    "tire_stiff":  dict(mu=0.50, m=1500.0, cs=0.25),                                    # cornering stiffness -75 %
    "sensor_bias": dict(mu=0.60, m=1500.0, bias={4: 0.2, 5: 0.08, 6: 0.02}),            # v_y bias 0.2 m/s, gyro bias 0.08 rad/s, steering-sensor bias 0.02 rad
    "steer_fault": dict(mu=0.70, m=1500.0, fault_t=3.0, steer_off=0.03, steer_gain=0.6),   # steering actuator degrades at t = 3 s: gain 0.6 and +0.03 rad offset
    "popout_wet":  dict(mu=0.55, m=1500.0, r_sense=12.0),                               # perception range 12 m (late detection)
}
def oracle_th(sc, t):
    """what the reference Oracle-MPC knows (single-track physics with the TRUE mu, mass, stiffness scale and CoG shift)"""
    return np.array([mu_at(sc, t) * ORACLE_MU_GAIN, V.m0 * sc["m"] / 1500.0, sc.get("cs", 1.0), sc.get("cog", 0.0)])

def make_world(seed):
    rng = np.random.default_rng(50_000 + seed)
    x1 = 50.0 + rng.uniform(-2, 2); x2 = 86.0 + rng.uniform(-2, 2)
    obs = np.array([[x1, path_y(x1) + 1.2 + rng.uniform(-0.15, 0.15)],  # O1 right of the road's left edge -> only the LEFT (+y) pass is inside the road
                    [x2, 1.9 + rng.uniform(-0.15, 0.15)]])          # O2 left of centre     -> only the RIGHT (-y) pass is inside the road
    v0 = V_REF + rng.uniform(-0.5, 0.5)
    n_steps = int(round(CFG["T_sim"] / DT))
    gust_jit = rng.uniform(-0.3, 0.3)
    meas_noise = rng.standard_normal((n_steps + 1, NX)) * NOISE_MEAS
    proc_noise = rng.standard_normal((n_steps + 1, NX)) * NOISE_PROC
    return obs, v0, meas_noise, proc_noise, gust_jit

## 6. The NMPC (CasADi/IPOPT) — one NLP structure for all controllers
* Decision variables: states `X`, inputs `U`, slacks for obstacle/road constraints.
* **Two-level constraints (C4).** *Physical* constraints `h(x_j) >= 0` (obstacle clearance, road edges) are always enforced (heavily penalised slack).
  The **conformal margin** is a *soft* second level: `h~_j = h(x_j) - q_j`,  `h~_j >= (1-gamma) h~_{j-1} - slack_m` (lightly penalised).
  `gamma = 1` is the classical tightened constraint `h~_j >= 0`; `gamma < 1` is the discrete-time CBF form (MPC-CBF).
  Why soft: far-horizon margins (up to metres under shift) can exceed the free corridor; making them hard destroys feasibility (we observed spin-outs), while the soft level keeps the *preference for clearance proportional to uncertainty* and never trades away physical safety.
* Dynamics model is switchable: **nominal**, **nominal + neural residual (Theta and the friction scale xi are runtime parameters, so adaptation never triggers a rebuild)**, or **oracle (true physics + true mu)**.

In [ ]:
BSEL = np.zeros((NX, ND)); BSEL[DYN, np.arange(ND)] = 1.0
W_M_LIN, W_M_QUAD = 20.0, 2.0     # soft-margin slack weights (physical constraints use 1e4 / 1e3)

class NMPC:
    def __init__(self, kind, margin, n=N):
        assert kind in ("nom", "nn", "oracle")
        self.kind, self.margin, self.n = kind, margin, n
        opti = ca.Opti()
        X = opti.variable(NX, n + 1); U = opti.variable(NU, n)
        S_ph = opti.variable(M_OBS + 2, n)                                   # slack of PHYSICAL constraints (heavily penalised)
        S_m = opti.variable(M_OBS + 2, n) if margin else None               # slack of conformal-MARGIN constraints (lightly penalised)
        p_s0 = opti.parameter(NX); p_uprev = opti.parameter(NU); p_obs = opti.parameter(M_OBS, 2)
        p_q = opti.parameter(n); p_gamma = opti.parameter(1)
        p_th = opti.parameter(4) if kind == "oracle" else None
        p_road = opti.parameter(2)                                          # [Y_LO, Y_HI] (scenario dependent)
        p_Th = opti.parameter(ND, FDIM) if kind == "nn" else None
        p_xi = opti.parameter(1) if kind in ("nom", "nn") else None
        p_tau = opti.parameter(1) if kind == "nn" else None                 # trust in the neural residual (0 = nominal model only, 1 = full residual)
        th0 = ca.vertcat(p_xi, 0.0) if p_xi is not None else None
        opti.subject_to(X[:, 0] == p_s0)
        for j in range(n):
            x, u = X[:, j], U[:, j]
            if kind == "nom":
                xn = F_nom(x, u, th0)
            elif kind == "oracle":
                xn = F_true_mpc(x, u, p_th)
            else:
                z = ca.vertcat(x[3:8], u)
                xn = F_nom(x, u, th0) + p_tau * ca.mtimes(ca.DM(BSEL), ca.mtimes(p_Th, phi_ca(z)) * ca.DM(y_sd))
            opti.subject_to(X[:, j + 1] == xn)
        opti.subject_to(opti.bounded(-V.d_max, ca.vec(U[0, :]), V.d_max))
        opti.subject_to(opti.bounded(V.ax_min, ca.vec(U[1, :]), V.ax_max))
        opti.subject_to(opti.bounded(-0.5, ca.vec(X[6, :]), 0.5))
        opti.subject_to(opti.bounded(1.0, ca.vec(X[3, :]), 30.0))
        opti.subject_to(opti.bounded(-1.3, ca.vec(X[2, :]), 1.3))
        opti.subject_to(opti.bounded(-8.0, ca.vec(X[4, :]), 8.0))
        opti.subject_to(opti.bounded(-1.5, ca.vec(X[5, :]), 1.5))
        opti.subject_to(ca.vec(S_ph) >= 0)
        if margin: opti.subject_to(ca.vec(S_m) >= 0)

        def barriers(px, py, q):
            hs = []
            for m in range(M_OBS):
                hs.append(ca.sqrt((px - p_obs[m, 0]) ** 2 + (py - p_obs[m, 1]) ** 2 + 1e-4) - (R_O + R_V + R_BUF) - q)
            hs.append(py - (p_road[0] + R_V) - q)
            hs.append((p_road[1] - R_V) - py - q)
            return hs
        h0 = barriers(X[0, 0], X[1, 0], 0.0); h_prev = h0
        for j in range(1, n + 1):
            hp = barriers(X[0, j], X[1, j], 0.0)
            for i in range(M_OBS + 2):
                opti.subject_to(hp[i] >= -S_ph[i, j - 1])                      # physical constraint (hard form)
            if margin:
                hj = barriers(X[0, j], X[1, j], p_q[j - 1])
                for i in range(M_OBS + 2):
                    opti.subject_to(hj[i] >= (1 - p_gamma) * h_prev[i] - S_m[i, j - 1])   # CBF-form conformal margin (soft)
                h_prev = hj
        cost = 0
        for j in range(1, n + 1):
            w = 3.0 if j == n else 1.0
            xj = X[0, j]; yref = PATH_A * ca.sin(_KX * xj); dyref = PATH_A * _KX * ca.cos(_KX * xj)
            psiref = ca.atan(dyref); rref = X[3, j] * (-PATH_A * _KX ** 2 * ca.sin(_KX * xj)) / (1 + dyref ** 2) ** 1.5
            cost += w * (1.0 * (X[1, j] - yref) ** 2 + 4.0 * (X[2, j] - psiref) ** 2 + 0.5 * (X[3, j] - V_REF) ** 2 + 1.0 * (X[5, j] - rref) ** 2)
        for j in range(n):
            du = U[:, j] - (p_uprev if j == 0 else U[:, j - 1])
            cost += 40.0 * U[0, j] ** 2 + 0.2 * U[1, j] ** 2 + 3000.0 * du[0] ** 2 + 1.0 * du[1] ** 2
        cost += 1e4 * ca.sum1(ca.sum2(S_ph)) + 1e3 * ca.sumsqr(S_ph)
        if margin: cost += W_M_LIN * ca.sum1(ca.sum2(S_m)) + W_M_QUAD * ca.sumsqr(S_m)
        opti.minimize(cost)
        opti.solver("ipopt", {"expand": True, "print_time": False, "ipopt.print_level": 0, "ipopt.sb": "yes",
                              "ipopt.max_iter": 60, "ipopt.tol": 1e-4, "ipopt.acceptable_tol": 1e-2,
                              "ipopt.acceptable_iter": 3, "ipopt.mu_strategy": "adaptive"})
        self.opti, self.X, self.U, self.S_ph, self.S_m = opti, X, U, S_ph, S_m
        self.p = dict(s0=p_s0, uprev=p_uprev, obs=p_obs, q=p_q, gamma=p_gamma, th=p_th, Th=p_Th, xi=p_xi, road=p_road, tau=p_tau)
        self.reset()

    def reset(self):
        self.X0, self.U0 = None, None

    def solve(self, s0, uprev, obs, q, gamma, th=None, Theta=None, xi=1.0, road=(Y_LO, Y_HI), tau=1.0):
        o, p = self.opti, self.p
        o.set_value(p["road"], np.asarray(road, float))
        o.set_value(p["s0"], s0); o.set_value(p["uprev"], uprev); o.set_value(p["obs"], obs)
        o.set_value(p["q"], q); o.set_value(p["gamma"], gamma)
        if p["th"] is not None: o.set_value(p["th"], th)
        if p["Th"] is not None: o.set_value(p["Th"], Theta)
        if p["xi"] is not None: o.set_value(p["xi"], xi)
        if p["tau"] is not None: o.set_value(p["tau"], tau)
        if self.X0 is None:
            Xg = np.tile(s0[:, None], (1, self.n + 1)); Ug = np.tile(uprev[:, None], (1, self.n))
        else:
            Xg = np.hstack([self.X0[:, 1:], self.X0[:, -1:]]); Xg[:, 0] = s0; Ug = np.hstack([self.U0[:, 1:], self.U0[:, -1:]])
        o.set_initial(self.X, Xg); o.set_initial(self.U, Ug); o.set_initial(self.S_ph, 0)
        if self.margin: o.set_initial(self.S_m, 0)
        t0 = time.perf_counter(); ok = True
        def grab(get):
            sl = float(np.sum(get(self.S_ph))); sm = float(np.sum(get(self.S_m))) if self.margin else 0.0
            return get(self.X), get(self.U), sl, sm
        try:
            sol = o.solve(); Xs, Us, sl, sm = grab(sol.value)
        except RuntimeError:
            ok = False
            try: Xs, Us, sl, sm = grab(o.debug.value)
            except Exception: Xs, Us, sl, sm = Xg, Ug, np.nan, np.nan
        dt = time.perf_counter() - t0
        Xs = np.asarray(Xs).reshape(NX, self.n + 1); Us = np.asarray(Us).reshape(NU, self.n)
        if not np.all(np.isfinite(Xs)) or not np.all(np.isfinite(Us)):
            Xs, Us, ok = Xg, Ug, False
        self.X0, self.U0 = Xs, Us
        return dict(u=Us[:, 0].copy(), X=Xs, U=Us, ok=ok, time=dt, slack=sl, slack_m=sm)

t0 = time.time()
MPCS = {("nom", False): NMPC("nom", False), ("nom", True): NMPC("nom", True), ("nn", False): NMPC("nn", False),
        ("nn", True): NMPC("nn", True), ("oracle", False): NMPC("oracle", False)}
print(f"NMPC built (5 variants) in {time.time()-t0:.1f}s")

## 7. Online learning and uncertainty quantification
### 7.1 Online friction-scale identification (C1)
Two estimators of the friction scale `xi`: a scalar **RLS** with forgetting (linear-in-`xi` regression of the v_y and r channels) and a **moving-horizon estimator** (nonlinear least squares over a grid of `xi`, window of 8 transitions).
### 7.2 Model-side multi-step scores
At time `t` and for every horizon `j = 1..N` we re-roll the *current* model from the measured state at `t-j` along the *executed* inputs and compare with the measured position at `t`. These scores depend only on model error — not on what the controller planned — which keeps them (approximately) exchangeable across controllers.
### 7.3 Horizon-grouped, normalised ACI with shift propagation (C2, C3)
* Normalise scores by the in-distribution median growth curve `g_j`.
* Three horizon groups `[1-4], [5-10], [11-25]`, each with an ACI-controlled quantile `kappa_g` (`alpha_{t+1} = alpha_t + eta (alpha - err_t)`).
* **Propagation:** `kappa_g >= kappa_g^cal * infl_{g-1}`: inflation seen at short horizons (observed after 1-4 steps) is imposed on longer horizons (observed after up to 25 steps).
* Tube: `q_j = min(q_cap, kappa_{g(j)} * g_j)`.
### 7.4 Joint coverage (C5) and the other tube baselines
* Per-step coverage of 90 % does **not** give 90 % *joint* coverage over the 25 steps. `CA-NMPC-J` multiplies the tube by `lambda >= lambda_cal`, where `lambda_cal` is the offline split-conformal quantile (level 0.9) of
  `max_j  e_j / (kappa_G(j) g_j)` over full-horizon calibration origins, and `lambda` is then driven online by the delayed joint-miss indicator (multiplicative ACI-type update).
* `static`: split-conformal tube, never updated. `const`: `q_j = c * g_j`, `c` grid-tuned on development seeds. `ens`: ensemble spread (uncalibrated).

In [ ]:
XI_CFG = dict(lam=0.93, p0=1.0, p_max=3.0, bounds=(0.12, 1.3), w=(1 / 0.05, 1 / 0.005))
class XiEstimator:
    """Scalar RLS (with forgetting) for the friction scale xi.
    Tyre forces are exactly linear in xi, so the one-step residual w.r.t. the xi=1 model is (xi-1)*g(s,u) with g the one-step tyre-force sensitivity
    (computed from the model by finite differences). We regress the lateral (v_y) and yaw-rate (r) channels, weighted by their sensor noise."""
    def __init__(self, Theta0, **kw):
        c = dict(XI_CFG); c.update(kw); self.c = c; self.Theta0 = Theta0
        self.theta = 0.0; self.P = c['p0']; self.xi = 1.0; self.last = 0.0
    def update(self, s_prev, u_prev, s_now, tau=1.0):
        c = self.c; S = s_prev[None]; U = u_prev[None]
        th1 = np.array([[1.0, 0.0]]); th2 = np.array([[1.05, 0.0]])
        base = np_step(F_nom, S, U, th1)[0]
        if self.Theta0 is not None: base[DYN] += tau * resid_np(np.hstack([S[:, DYN], U]), self.Theta0)[0]
        g = (np_step(F_nom, S, U, th2)[0] - np_step(F_nom, S, U, th1)[0]) / 0.05
        res = 0.0
        for ch, w in zip((4, 5), c['w']):
            a = g[ch] * w; y = (s_now[ch] - base[ch]) * w
            K = self.P * a / (c['lam'] + a * a * self.P)
            self.theta += K * (y - a * self.theta); self.P = min(c['p_max'], (self.P - K * a * self.P) / c['lam'])
            res += abs(y - a * self.theta)
        self.xi = float(np.clip(1.0 + self.theta, *c['bounds'])); self.theta = self.xi - 1.0
        self.last = res / 2.0
        return self.last

class XiMHE:
    """Moving-horizon estimator of the friction scale xi (a stronger, *nonlinear* classical alternative to the scalar RLS):
    over the last W transitions it minimises the weighted one-step prediction error of v_y and r of the (nominal [+ neural residual]) model on a log-spaced grid
    of xi values (no linearisation, handles tyre saturation), plus a weak arrival-cost prior towards the previous estimate; the minimum is refined by a parabola."""
    def __init__(self, Theta0, W=8, K=29, prior=2.0, bounds=(0.12, 1.3), w=(1 / 0.05, 1 / 0.005)):
        self.Theta0, self.W, self.prior, self.w = Theta0, W, prior, w
        self.grid = np.exp(np.linspace(np.log(bounds[0]), np.log(bounds[1]), K)); self.hist = deque(maxlen=W); self.xi = 1.0; self.last = 0.0
    def update(self, s_prev, u_prev, s_now, tau=1.0):
        self.hist.append((s_prev, u_prev, s_now)); H = list(self.hist); Wn, K = len(H), len(self.grid)
        S0 = np.array([h[0] for h in H]); U0 = np.array([h[1] for h in H]); Y0 = np.array([h[2] for h in H])
        S = np.tile(S0, (K, 1)); U = np.tile(U0, (K, 1)); Y = np.tile(Y0, (K, 1))
        TH = np.repeat(np.stack([self.grid, np.zeros(K)], 1), Wn, axis=0)
        pred = np_step(F_nom, S, U, TH)
        if self.Theta0 is not None: pred[:, DYN] += tau * resid_np(np.hstack([S[:, DYN], U]), self.Theta0)
        err = ((pred[:, 4] - Y[:, 4]) * self.w[0]) ** 2 + ((pred[:, 5] - Y[:, 5]) * self.w[1]) ** 2
        cost = err.reshape(K, Wn).sum(1) / Wn + self.prior * (np.log(self.grid / self.xi) / 0.35) ** 2 / Wn
        i = int(np.argmin(cost)); xi = self.grid[i]
        if 0 < i < K - 1:                                           # parabolic refinement in log-xi
            y0, y1, y2 = cost[i - 1], cost[i], cost[i + 1]; den = y0 - 2 * y1 + y2
            if den > 1e-12: xi = float(np.exp(np.log(self.grid[i]) + 0.5 * (y0 - y2) / den * (np.log(self.grid[i + 1]) - np.log(self.grid[i]))))
        self.xi = float(np.clip(xi, self.grid[0], self.grid[-1])); self.last = float(np.sqrt(cost[i]))
        return self.last

def rollout_np(S0, Useq, Theta, use_nn=True, member=None, xi=1.0, tau=1.0):
    """batched open-loop rollout. S0:(B,8) Useq:(B,L,2) -> (B,L+1,8)"""
    B, L = Useq.shape[0], Useq.shape[1]
    out = np.zeros((B, L + 1, NX)); out[:, 0] = S0; S = S0.copy()
    TH = np.tile([xi, 0.0], (B, 1))
    for l in range(L):
        Sn = np_step(F_nom, S, Useq[:, l], TH)
        if use_nn:
            Z = np.hstack([S[:, DYN], Useq[:, l]])
            if member is None: d = (phi_np(Z) @ Theta.T) * y_sd
            else: d = member_np(zn(Z), member) * y_sd
            Sn[:, DYN] += tau * d
        S = Sn; out[:, l + 1] = S
    return out

def multistep_scores(S_hist, U_hist, t, Theta, n=N, use_nn=True, xi=1.0, tau=1.0):
    """e[j-1] = || model_j-step_pred(from t-j) - measured position at t ||, NaN if not yet available. S_hist has t+1 rows, U_hist t rows."""
    e = np.full(n, np.nan); B = min(n, t)
    if B < 1: return e
    S0 = np.array([S_hist[t - j] for j in range(1, B + 1)])
    Useq = np.zeros((B, B, NU))
    for j in range(1, B + 1): Useq[j - 1, :j] = np.array(U_hist[t - j:t])
    R = rollout_np(S0, Useq, Theta, use_nn, None, xi, tau)
    for j in range(1, B + 1): e[j - 1] = np.linalg.norm(R[j - 1, j, :2] - S_hist[t][:2])
    return e

# --- offline calibration scores (in-distribution, full online noise) with the OFFLINE model THETA0
def calibration_scores(trajs, Theta, n=N, use_nn=True):
    """returns (per-horizon pooled scores [list of n arrays], per-ORIGIN score matrix (n_origins, n) for origins with a full horizon -> needed for the joint calibration)"""
    sc = [[] for _ in range(n)]; Em = []
    for (S, U, mu, m) in trajs:
        T = len(U)
        for i in range(0, T - 1):
            L = min(n, T - i)
            R = rollout_np(S[i][None], U[i:i + L][None], Theta, use_nn)[0]
            row = np.full(n, np.nan)
            for j in range(1, L + 1):
                e = np.linalg.norm(R[j, :2] - S[i + j, :2]); sc[j - 1].append(e); row[j - 1] = e
            if L == n: Em.append(row)
    return [np.array(s) for s in sc], np.array(Em)
t0 = time.time()
CAL_SCORES, CAL_E = calibration_scores(traj_cal, THETA0)
CAL_SCORES_NOM, CAL_E_NOM = calibration_scores(traj_cal, None, use_nn=False)
print(f"calibration scores: {[len(s) for s in CAL_SCORES][:3]}... in {time.time()-t0:.1f}s | median/95% position error at j=1,5,10,25:",
      [(round(float(np.median(CAL_SCORES[j-1])), 3), round(float(np.quantile(CAL_SCORES[j-1], .95)), 3)) for j in (1, 5, 10, 25)])

GROUPS = [(1, 4), (5, 10), (11, 25)]       # default horizon groups (also used for the coverage REPORT in every experiment)
GROUPS_REPORT = GROUPS
def group_of(j, groups=GROUPS):
    for gi, (a, b) in enumerate(groups):
        if a <= j <= b: return gi

class ConformalTube:
    """mode: 'static' (split CP, never updated) | 'const' (constant multiple c of the in-distribution median growth curve; c is GRID-TUNED on development seeds)
             | 'aci' (online grouped ACI, no propagation) | 'aci_prop' (ours: + shift propagation) | 'aci_joint' (ours + a global multiplier lambda driven by the *joint* coverage event)"""
    def __init__(self, cal_scores, mode="aci_prop", alpha=0.10, eta=0.05, q_cap=3.0, win_per_j=20, seed=0, q_floor=0.0, groups=None, c=1.0,
                 alpha_joint=0.10, eta_joint=0.15, lam_max=6.0, cal_E=None):
        self.mode, self.alpha, self.eta, self.q_cap, self.q_floor, self.c = mode, alpha, eta, q_cap, q_floor, c
        self.groups = groups or GROUPS
        n = len(cal_scores)
        self.g = np.maximum(np.array([np.median(s) for s in cal_scores]), 1e-3)
        rng = np.random.default_rng(seed)
        self.win, self.kappa_cal = [], np.zeros(len(self.groups))
        for gi, (a, b) in enumerate(self.groups):
            pool = np.concatenate([cal_scores[j - 1] / self.g[j - 1] for j in range(a, b + 1)])
            self.kappa_cal[gi] = np.quantile(pool, 1 - alpha)
            W = win_per_j * (b - a + 1)
            self.win.append(deque(rng.choice(pool, size=min(W, len(pool)), replace=False).tolist(), maxlen=W))
        self.kappa = self.kappa_cal.copy(); self.alpha_t = np.full(len(self.groups), alpha)
        self.n = n; self.lam = 1.0; self.alpha_joint, self.eta_joint, self.lam_max = alpha_joint, eta_joint, lam_max
        self.lam_cal = 1.0
        if mode == "aci_joint" and cal_E is not None and len(cal_E):
            # offline JOINT calibration: lambda such that, in-distribution, ALL horizon steps of an N-step prediction are covered with probability 1 - alpha_joint
            ratio = np.stack([cal_E[:, j - 1] / self.g[j - 1] / self.kappa_cal[group_of(j, self.groups)] for j in range(1, n + 1)], 1).max(1)
            self.lam_cal = float(max(1.0, np.quantile(ratio, 1 - alpha_joint))); self.lam = self.lam_cal
    def margins(self):
        if self.mode == "const":
            return np.minimum(self.q_cap, self.c * self.g)
        q = np.array([min(self.q_cap, self.lam * self.kappa[group_of(j, self.groups)] * self.g[j - 1]) for j in range(1, self.n + 1)])
        return np.maximum(q, self.q_floor)
    def update_joint(self, miss):
        """delayed feedback of the JOINT event (an N-step prediction made N steps ago was not fully covered): multiplicative ACI-type recursion on a global inflation lambda in [1, lam_max]"""
        if self.mode != "aci_joint": return
        self.lam = float(np.clip(self.lam * np.exp(self.eta_joint * (float(miss) - self.alpha_joint)), self.lam_cal, self.lam_max))
    def update(self, e):
        if self.mode in ("static", "const"): return
        for gi, (a, b) in enumerate(self.groups):
            s = e[a - 1:b]; s = s[~np.isnan(s)] / self.g[a - 1:b][~np.isnan(e[a - 1:b])]
            if len(s) == 0: continue
            err = float(np.mean(s > self.kappa[gi]))
            self.alpha_t[gi] = np.clip(self.alpha_t[gi] + self.eta * (self.alpha - err), 0.005, 0.5)
            self.win[gi].extend(s.tolist())
            self.kappa[gi] = float(np.quantile(np.array(self.win[gi]), 1 - self.alpha_t[gi]))
        if self.mode in ("aci_prop", "aci_joint"):
            infl = self.kappa / self.kappa_cal
            for gi in range(1, len(self.groups)):
                self.kappa[gi] = max(self.kappa[gi], self.kappa_cal[gi] * max(1.0, infl[gi - 1]))
                infl[gi] = self.kappa[gi] / self.kappa_cal[gi]

def ensemble_margins(s_now, U_plan, kappa_e=3.0, q_cap=3.0, n=N, xi=1.0, tau=1.0):
    """uncalibrated baseline: spread of the K ensemble members' predicted positions along the planned inputs"""
    Useq = np.tile(U_plan.T[None], (K_ENS, 1, 1))
    R = np.stack([rollout_np(s_now[None], Useq[k:k + 1], None, True, member=k, xi=xi, tau=tau)[0] for k in range(K_ENS)])  # (K, n+1, 8)
    P = R[:, 1:, :2]; sig = np.linalg.norm(P - P.mean(0, keepdims=True), axis=2).max(0)
    return np.clip(np.nan_to_num(kappa_e * sig, nan=q_cap, posinf=q_cap), 0.0, q_cap)
class ResidualGate:
    """Online TRUST in the neural residual (C1b). The residual was learned on dry-road data only, so under shift it can be *worse* than no residual.
    At every step we compare, for two horizons, the position error of the model WITH the residual and of the nominal model (both with the current xi) against the measurement;
    an exponentially weighted relative improvement  r = (eA - eB)/(eA + eB + eps)  is mapped to a trust  tau = clip(r_bar / r0, 0, 1)  that scales the residual inside the MPC, the scores and the estimator."""
    def __init__(self, Theta, rho=0.9, r0=0.15, js=(4, 8), eps=0.03):
        self.Theta, self.rho, self.r0, self.js, self.eps = Theta, rho, r0, js, eps; self.rbar = r0; self.tau = 1.0
    def update(self, S_hist, U_hist, k, xi):
        js = [j for j in self.js if k - j >= 0]
        if not js: return self.tau
        B = len(js); S0 = np.array([S_hist[k - j] for j in js]); Useq = np.zeros((B, max(js), NU))
        for b, j in enumerate(js): Useq[b, :j] = np.array(U_hist[k - j:k])
        RA = rollout_np(S0, Useq, None, False, None, xi, 0.0); RB = rollout_np(S0, Useq, self.Theta, True, None, xi, 1.0)
        r = []
        for b, j in enumerate(js):
            eA = np.linalg.norm(RA[b, j, :2] - S_hist[k][:2]); eB = np.linalg.norm(RB[b, j, :2] - S_hist[k][:2]); r.append((eA - eB) / (eA + eB + self.eps))
        self.rbar = self.rho * self.rbar + (1 - self.rho) * float(np.mean(r)); self.tau = float(np.clip(self.rbar / self.r0, 0.0, 1.0))
        return self.tau

print("online-learning components ready")

## 8. Controllers (baselines, proposed method, ablations) and the closed-loop simulator
All use the same NLP structure; they differ in *model*, *online friction identification*, *tube* and *constraint form*:

| name | model | friction ID | tube | constraints |
|---|---|---|---|---|
| Nominal-MPC | nominal (assumes dry road) | – | – | hard |
| Adaptive-MPC (FricID) | nominal + scalar RLS for `xi` (classical adaptive MPC) | RLS | – | hard |
| Adaptive-MPC (MHE) | nominal + **moving-horizon (nonlinear LS) estimate of `xi`** | MHE | – | hard |
| Neural-MPC | nominal + offline NN residual | – | – | hard |
| Neural-MPC+FricID | + RLS friction identification | RLS | – | hard |
| **Neural-MPC+FricID+Gate** | + **online trust gate** on the neural residual (C1b) | RLS | – | hard |
| Nominal+ACI (tube only) | nominal (no learning) | – | grouped ACI + propagation | CBF-form |
| **Adaptive+ACI (no NN)** | nominal + RLS (isolates the tube from the NN) | RLS | grouped ACI + propagation | CBF-form |
| NN+FricID+EnsTube | + RLS | RLS | ensemble-spread tube (uncalibrated) | CBF-form |
| NN+FricID+StaticCP | + RLS | RLS | split-conformal tube, fixed after offline calibration | CBF-form |
| **NN+FricID+TunedTube** | + RLS | RLS | **constant tube, multiplier grid-tuned on development seeds** (robust-MPC-like) | CBF-form |
| **CA-NMPC (ours)** | + RLS | RLS | grouped ACI + shift propagation | CBF-form |
| **CA-NMPC-J (ours, joint)** | + RLS | RLS | + global multiplier driven by the *joint* N-step coverage event | CBF-form |
| ablation: w/o propagation | | RLS | grouped ACI only | CBF-form |
| ablation: w/o residual gate | NN ungated | RLS | grouped ACI + propagation | CBF-form |
| ablation: w/o FricID | NN offline only | – | grouped ACI + propagation | CBF-form |
| ablation: hard constraints | | RLS | grouped ACI + propagation | hard (gamma = 1) |
| Oracle-MPC (reference only) | single-track physics with the true mu, mass, stiffness, CoG | – | – | hard |

**Closed-loop timing.** With `DELAY = 1` (default) the command computed from the measurement at step `k` is applied at step `k+1` (the solver needs time); the MPC is started from the *model-predicted* state at `k+1`
(delay compensation) and the margins are shifted by one step accordingly. All controllers have the same delay. Measured solve times are reported separately (`deadline_miss` = fraction of solves that took longer than one control period on *this* machine).

In [ ]:
ALPHA, ETA, Q_CAP, GAMMA = 0.10, 0.05, 3.0, 0.5
SPECS = {
    "Nominal-MPC":              dict(model="nom",    adapt=False,  tube=None,        gamma=1.0),
    "Adaptive-MPC (FricID)":    dict(model="nom",    adapt="rls",  tube=None,        gamma=1.0),
    "Adaptive-MPC (MHE)":       dict(model="nom",    adapt="mhe",  tube=None,        gamma=1.0),
    "Neural-MPC":               dict(model="nn",     adapt=False,  tube=None,        gamma=1.0),
    "Neural-MPC+FricID":        dict(model="nn",     adapt="rls",  tube=None,        gamma=1.0),
    "Neural-MPC+FricID+Gate":   dict(model="nn",     adapt="rls",  tube=None,        gamma=1.0, gate=True),
    "Nominal+ACI (tube only)":  dict(model="nom",    adapt=False,  tube="aci_prop",  gamma=GAMMA),
    "Adaptive+ACI (no NN)":     dict(model="nom",    adapt="rls",  tube="aci_prop",  gamma=GAMMA),
    "NN+FricID+EnsTube":        dict(model="nn",     adapt="rls",  tube="ens",       gamma=GAMMA, gate=True),
    "NN+FricID+StaticCP":       dict(model="nn",     adapt="rls",  tube="static",    gamma=GAMMA, gate=True),
    "NN+FricID+TunedTube":      dict(model="nn",     adapt="rls",  tube="const",     gamma=GAMMA, gate=True, c=1.0),
    "CA-NMPC (ours)":           dict(model="nn",     adapt="rls",  tube="aci_prop",  gamma=GAMMA, gate=True),
    "CA-NMPC-J (ours, joint)":  dict(model="nn",     adapt="rls",  tube="aci_joint", gamma=GAMMA, gate=True),
    "abl: w/o propagation":     dict(model="nn",     adapt="rls",  tube="aci",       gamma=GAMMA, gate=True),
    "abl: w/o residual gate":   dict(model="nn",     adapt="rls",  tube="aci_prop",  gamma=GAMMA),
    "abl: w/o FricID":          dict(model="nn",     adapt=False,  tube="aci_prop",  gamma=GAMMA, gate=True),
    "abl: hard constraints":    dict(model="nn",     adapt="rls",  tube="aci_prop",  gamma=1.0,   gate=True),
    "Oracle-MPC (ref.)":        dict(model="oracle", adapt=False,  tube=None,        gamma=1.0),
}
if os.environ.get("CA_CTRLS"):
    _keep = os.environ["CA_CTRLS"].split(";"); SPECS = {k: v for k, v in SPECS.items() if k in _keep}
OURS, OURS_J, ADAPT_RLS, ADAPT_MHE = "CA-NMPC (ours)", "CA-NMPC-J (ours, joint)", "Adaptive-MPC (FricID)", "Adaptive-MPC (MHE)"

def clearance_series(P, obs):
    """min clearance over time incl. linear interpolation between samples; P:(T,2)"""
    cl = np.full(len(P), np.inf)
    for t in range(len(P) - 1):
        for a in np.linspace(0, 1, 6):
            p = (1 - a) * P[t] + a * P[t + 1]
            c = np.min(np.linalg.norm(obs - p, axis=1) - R_O - R_V); cl[t] = min(cl[t], c)
    cl[-1] = np.min(np.linalg.norm(obs - P[-1], axis=1) - R_O - R_V)
    return cl

def joint_miss(E, Q, t, groups=None, n=N, tol=1e-9):
    """Was the N-step prediction issued at step t (margins Q[t]) NOT fully covered? E[k, j-1] = score of horizon j observed at step k (origin k-j)."""
    j = np.arange(1, n + 1); miss = E[t + j, j - 1] > Q[t] + tol           # NaN -> False
    if groups is None: return bool(miss.any())
    return [bool(miss[a - 1:b].any()) for (a, b) in groups]

def predict_next(sm, u, kind, Theta, xi, sc, t, tau=1.0):
    S, U = sm[None], u[None]
    if kind == "oracle": return np_step(F_true_mpc, S, U, oracle_th(sc, t)[None])[0]
    sn = np_step(F_nom, S, U, np.array([[xi, 0.0]]))[0]
    if kind == "nn": sn[DYN] += tau * resid_np(np.hstack([S[:, DYN], U]), Theta)[0]
    return sn

def make_tube(spec, use_nn, seed):
    return ConformalTube(CAL_SCORES if use_nn else CAL_SCORES_NOM, mode=spec["tube"], alpha=spec.get("alpha", ALPHA), eta=spec.get("eta", ETA),
                         q_cap=spec.get("q_cap", Q_CAP), groups=spec.get("groups"), win_per_j=spec.get("win", 20), c=spec.get("c", 1.0), seed=seed,
                         eta_joint=spec.get("eta_joint", 0.15), lam_max=spec.get("lam_max", 6.0), cal_E=(CAL_E if use_nn else CAL_E_NOM))

def simulate(name, spec, scen_name, seed, keep=False, delay=None):
    delay = DELAY if delay is None else delay
    sc = SCEN[scen_name]; obs, v0, mn, pn, gj = make_world(seed)
    plant = make_plant(sc); s = plant.reset(v0)
    ylo, yhi, rsense = sc.get("ylo", Y_LO), sc.get("yhi", Y_HI), sc.get("r_sense", R_SENSE)
    bias = np.zeros(NX)
    for i_, v_ in sc.get("bias", {}).items(): bias[i_] = v_
    kind = spec["model"]; mpc = MPCS[(kind, spec["tube"] is not None)]; mpc.reset()
    use_nn = kind == "nn"; Theta = THETA0; xi = 1.0; tau = 1.0
    gate = ResidualGate(THETA0) if (use_nn and spec.get("gate")) else None
    est = None
    if spec["adapt"] == "rls": est = XiEstimator(THETA0 if use_nn else None)
    elif spec["adapt"] == "mhe": est = XiMHE(THETA0 if use_nn else None)
    tube = make_tube(spec, use_nn, seed) if spec["tube"] in ("static", "const", "aci", "aci_prop", "aci_joint") else None
    need_scores = spec["tube"] is not None
    n_steps = int(round(CFG["T_sim"] / DT))
    Sm, Uh, Str = [], [], [s.copy()]
    u_pending = np.zeros(2)
    E = np.full((n_steps + 2, N), np.nan); Q = np.full((n_steps + 2, N), np.nan)
    q_prev = np.zeros(N); q_log, cov_log, ok_log, t_log, slack_log, slackm_log, innov_log, xi_log, lam_log, tau_log = [], [], [], [], [], [], [], [], [], []
    terminated = finished = False
    for k in range(n_steps):
        sm = s + bias + mn[k]; Sm.append(sm)
        if k >= 1:
            if est is not None:
                innov_log.append(est.update(Sm[k - 1], Uh[k - 1], sm, tau)); xi = est.xi
            if gate is not None: tau = gate.update(Sm, Uh, k, xi)
            if need_scores:
                e = multistep_scores(Sm, Uh, k, Theta if use_nn else None, use_nn=use_nn, xi=xi, tau=tau)
                E[k] = e
                cov_log.append(np.where(np.isnan(e), np.nan, (e <= q_prev + 1e-9).astype(float)))
                if tube is not None:
                    tube.update(e)
                    if k >= N: tube.update_joint(joint_miss(E, Q, k - N))
        # ---- margins
        if spec["tube"] is None: q = np.zeros(N)
        elif spec["tube"] == "ens":
            Uplan = mpc.U0 if mpc.U0 is not None else np.zeros((NU, N))
            q = ensemble_margins(sm, Uplan, xi=xi, tau=tau)
        else: q = tube.margins()
        q = np.clip(np.nan_to_num(q, nan=Q_CAP, posinf=Q_CAP), 0.0, Q_CAP); q_prev = q.copy(); Q[k] = q
        q_plan = np.concatenate([q[1:], q[-1:]]) if delay else q          # with a one-step delay the plan starts one step later than the measurement
        obs_vis = np.where((obs[:, 0] - sm[0] < rsense)[:, None] & (obs[:, 0] - sm[0] > -5.0)[:, None], obs, np.array([[sm[0] + 500.0, 100.0]]))
        # ---- solve (possibly from the model-predicted state when there is a computational delay)
        if delay == 0:
            s_plan, u_prev_plan, t_plan = sm, (Uh[-1] if Uh else np.zeros(2)), k * DT; u_cur = None
        else:
            u_cur = u_pending; s_plan = predict_next(sm, u_cur, kind, Theta, xi, sc, k * DT, tau); u_prev_plan, t_plan = u_cur, (k + 1) * DT
        res = mpc.solve(s_plan, u_prev_plan, obs_vis, q_plan, spec["gamma"], th=(oracle_th(sc, t_plan) if kind == "oracle" else None), Theta=Theta, xi=xi, road=(ylo, yhi), tau=tau)
        u_new = np.array([np.clip(res["u"][0], -V.d_max, V.d_max), np.clip(res["u"][1], V.ax_min, V.ax_max)])
        if delay == 0: u_cur = u_new
        else: u_pending = u_new
        xi_log.append(xi); tau_log.append(tau); lam_log.append(getattr(tube, "lam", 1.0) if tube is not None else 1.0)
        t_log.append(res["time"]); ok_log.append(res["ok"]); slack_log.append(res["slack"]); slackm_log.append(res["slack_m"]); q_log.append(q)
        # ---- plant
        gust = sc["gust_a"] * DT if ("gust_t0" in sc and sc["gust_t0"] + gj <= k * DT < sc["gust_t0"] + gj + sc["gust_dur"]) else 0.0
        s = plant.step(u_cur, k * DT, pn[k], gust); Uh.append(u_cur); Str.append(s.copy())
        if abs(s[2]) > 1.2 or s[3] < 1.5 or abs(s[4]) > 6.0 or not np.all(np.isfinite(s)):
            terminated = True; break
        if s[0] >= X_END: finished = True; break
    Str = np.array(Str); P = Str[:, :2]; Uh_a = np.array(Uh)
    cl = clearance_series(P, obs); min_clear = float(cl.min())
    road_viol = float(max(0.0, np.max((ylo + R_V) - P[:, 1]), np.max(P[:, 1] - (yhi - R_V))))
    collision = min_clear < 0; road = road_viol > 0.05
    cov = np.array(cov_log) if cov_log else np.zeros((0, N))
    cov_g = [float(np.nanmean(cov[:, a - 1:b])) if cov.size and np.any(~np.isnan(cov[:, a - 1:b])) else np.nan for (a, b) in GROUPS_REPORT]
    # joint coverage of the N-step predictions that were fully resolved before the episode ended
    L = len(Sm); jm, jg = [], []
    if need_scores:
        for t0_ in range(0, L - N):
            r_ = joint_miss(E, Q, t0_, GROUPS_REPORT); jm.append(any(r_)); jg.append(r_)
    cov_joint = float(1.0 - np.mean(jm)) if jm else np.nan
    # per-step coverage of the margins that were ISSUED at the origin (score of horizon j observed at step k belongs to the origin k-j, whose margin is Q[k-j]); `cov_g*` above compares with the previous step's margin
    cov_iss = np.full((len(Sm), N), np.nan)
    if need_scores:
        for j_ in range(1, N + 1):
            ks_ = np.arange(j_, len(Sm)); e_ = E[ks_, j_ - 1]; q_ = Q[ks_ - j_, j_ - 1]
            cov_iss[ks_, j_ - 1] = np.where(np.isnan(e_) | np.isnan(q_), np.nan, (e_ <= q_ + 1e-9).astype(float))
    cov_iss_g = [float(np.nanmean(cov_iss[:, a - 1:b])) if np.any(~np.isnan(cov_iss[:, a - 1:b])) else np.nan for (a, b) in GROUPS_REPORT]
    cov_joint_g = [float(1.0 - np.mean([x[g] for x in jg])) if jg else np.nan for g in range(len(GROUPS_REPORT))]
    qlog = np.array(q_log); ay = Str[:, 3] * Str[:, 5]
    jerk = np.diff(Str[:, 7]) / DT; srate = np.diff(Uh_a[:, 0]) / DT if len(Uh_a) > 1 else np.zeros(1)
    xi_true = np.array([mu_at(sc, i_ * DT) / 0.9 for i_ in range(len(xi_log))])
    reach = np.where(P[:, 0] >= 95.0)[0]
    m = dict(controller=name, scenario=scen_name, seed=seed,
             success=float((not collision) and (not road) and (not terminated)), collision=float(collision), road_violation=float(road),
             terminated=float(terminated), finished=float(finished), min_clearance=min_clear, road_viol_m=road_viol,
             rms_y=float(np.sqrt(np.mean((P[:, 1] - path_y(P[:, 0])) ** 2))), mean_speed=float(Str[:, 3].mean()), progress=float(P[-1, 0]),
             t95=float(reach[0] * DT) if len(reach) else np.nan,
             ay_rms=float(np.sqrt(np.mean(ay ** 2))), ay_max=float(np.abs(ay).max()), jerk_rms=float(np.sqrt(np.mean(jerk ** 2))), steer_rate_rms=float(np.sqrt(np.mean(srate ** 2))),
             solve_ms=1e3 * float(np.mean(t_log)), solve_ms_p95=1e3 * float(np.quantile(t_log, 0.95)), deadline_miss=float(np.mean(np.array(t_log) > DT)),
             solver_fail=float(1 - np.mean(ok_log)), slack_mean=float(np.nanmean(slack_log)), slack_margin=float(np.nanmean(slackm_log)),
             q_mid=float(qlog[:, 11].mean()), q_end=float(qlog[:, -1].mean()), lam_end=float(lam_log[-1]), tau_mean=float(np.mean(tau_log)), tau_end=float(tau_log[-1]),
             xi_rmse=float(np.sqrt(np.mean((np.array(xi_log) - xi_true) ** 2))) if (est is not None and PLANT == "home") else np.nan,
             cov_overall=float(np.nanmean(cov)) if cov.size and np.any(~np.isnan(cov)) else np.nan,
             cov_g1=cov_g[0], cov_g2=cov_g[1], cov_g3=cov_g[2], cov_iss_g1=cov_iss_g[0], cov_iss_g2=cov_iss_g[1], cov_iss_g3=cov_iss_g[2], cov_iss_overall=float(np.nanmean(cov_iss)) if np.any(~np.isnan(cov_iss)) else np.nan, cov_joint=cov_joint, cov_joint_g1=cov_joint_g[0], cov_joint_g2=cov_joint_g[1], cov_joint_g3=cov_joint_g[2],
             cov_j=(np.nanmean(cov, 0) if cov.size else np.full(N, np.nan)), q_j=qlog.mean(0))
    ex = dict(Str=Str, Uh=Uh_a, obs=obs, qlog=qlog, cov=cov, scen=scen_name, innov=np.array(innov_log), xi=np.array(xi_log), xi_true=xi_true, slack=np.array(slack_log), lam=np.array(lam_log), tau=np.array(tau_log)) if keep else None
    return m, ex
print("simulator ready")

## P1. Pilot configuration and pre-specified protocol
**Everything below is decided before any test seed is run.**
* `PILOT = "smoke"` (minutes, wiring check) · `"quick"` (8 test seeds, ~30-45 min) · `"pilot30"` (30 test seeds, ~2 h on a free 2-CPU Colab; resumable).
* **Seeds are disjoint by purpose:** `dev` seeds (0-3) → tune the baselines' free parameters and set scenario severity (baselines only); `diag` seeds (2000+) → Step 1 diagnosis and choosing the *fix*; `test` seeds (1000+) → the Step 2 comparison, never used for any decision.
* **Step 1** – why does CA-NMPC fail in `tire_stiff`? Controlled interventions (null tube, road-edge margins off, smaller `q_cap`, ...) on the diag seeds + an automatic verdict.
* **Step 2** – moving obstacles and obstacle-position error: does an **obstacle-side conformal margin** (online-calibrated, ACI, delayed feedback) beat **strong, fairly tuned baselines** (adaptive MPC, tuned velocity-uncertainty inflation, predictive CBF safety filter in two variants)?
* **Decision rule (pre-specified, applied automatically in the last cell):** path A (method paper) is "viable" only if the proposed controller beats **all four** strong baselines on the pooled shift scenarios by ≥ 10 pp with Holm-adjusted exact-McNemar p < 0.05. Otherwise the results are reported as they are and path B (evaluation/benchmark paper) is recommended.

In [ ]:
PILOT = os.environ.get("CA_PILOT", "quick")
_ALL_MO = ["mo_cv", "mo_swerve", "mo_stopgo", "mo_noisy", "mo_swerve_noisy", "mo_swerve_wet"]
PILOT_CFGS = {
    "smoke":   dict(seeds=[1000],                    diag=[2000, 2001],            dev=[0],             scen=["mo_cv", "mo_swerve"], infl_grid=(0.0, 0.5),               cbf_grid=(0.0,), cbf_gamma_grid=(0.3, 1.0)),
    "quick":   dict(seeds=list(range(1000, 1008)),   diag=list(range(2000, 2010)), dev=[0, 1, 2, 3],   scen=_ALL_MO,                infl_grid=(0.0, 0.15, 0.3, 0.5, 0.8), cbf_grid=(0.0, 0.5, 1.0), cbf_gamma_grid=(0.2, 0.3, 0.6, 1.0)),
    "pilot30": dict(seeds=list(range(1000, 1030)),   diag=list(range(2000, 2020)), dev=[0, 1, 2, 3],   scen=_ALL_MO,                infl_grid=(0.0, 0.15, 0.3, 0.5, 0.8), cbf_grid=(0.0, 0.5, 1.0), cbf_gamma_grid=(0.2, 0.3, 0.6, 1.0)),
}
PC = PILOT_CFGS[PILOT]
CFG["T_sim"] = 8.5                                   # full-length episodes (the vehicle must reach x = 100 m)
PO = f"{os.environ.get('CA_OUTROOT', '.')}/pilot_{PLANT}_{PILOT}"; os.makedirs(PO, exist_ok=True)
OBS_QCAP = 3.0                                       # cap of the obstacle-motion margin [m]
print(f"PILOT={PILOT} | plant={PLANT} | delay={DELAY} | test seeds={len(PC['seeds'])} | diag seeds={len(PC['diag'])} | dev seeds={PC['dev']} | outputs -> {PO}/")

## P2. NMPC with time-varying (moving) obstacle predictions and separate margins
Same NLP as Section 6, with three differences:
1. the obstacle positions are a **trajectory over the horizon** (constant-velocity prediction from the tracker, or the true future for the oracle reference);
2. margins are split in three: `q` (ego-model tube, soft CBF-form level; applies to obstacles and — optionally, `road_margin` — to the road edges), `qo` (obstacle-motion margin, soft level) and `qph` (obstacle inflation at the **physical** level: used by the tuned-inflation baseline and the hard conformal obstacle margin);
3. `road_margin=False` removes the ego-tube from the road-edge constraints (one of the Step 1 interventions).

In [ ]:
class NMPCT:
    def __init__(self, kind, margin, road_margin=True, n=N):
        assert kind in ("nom", "nn", "oracle")
        self.kind, self.margin, self.road_margin, self.n = kind, margin, road_margin, n
        opti = ca.Opti()
        X = opti.variable(NX, n + 1); U = opti.variable(NU, n)
        S_ph = opti.variable(M_OBS + 2, n); S_m = opti.variable(M_OBS + 2, n) if margin else None
        p_s0 = opti.parameter(NX); p_uprev = opti.parameter(NU); p_obs = opti.parameter(2 * M_OBS, n + 1)
        p_q = opti.parameter(n); p_qo = opti.parameter(M_OBS, n); p_qph = opti.parameter(M_OBS, n); p_gamma = opti.parameter(1)
        p_th = opti.parameter(4) if kind == "oracle" else None
        p_road = opti.parameter(2)
        p_Th = opti.parameter(ND, FDIM) if kind == "nn" else None
        p_xi = opti.parameter(1) if kind in ("nom", "nn") else None
        p_tau = opti.parameter(1) if kind == "nn" else None
        th0 = ca.vertcat(p_xi, 0.0) if p_xi is not None else None
        opti.subject_to(X[:, 0] == p_s0)
        for j in range(n):
            x, u = X[:, j], U[:, j]
            if kind == "nom": xn = F_nom(x, u, th0)
            elif kind == "oracle": xn = F_true_mpc(x, u, p_th)
            else:
                z = ca.vertcat(x[3:8], u)
                xn = F_nom(x, u, th0) + p_tau * ca.mtimes(ca.DM(BSEL), ca.mtimes(p_Th, phi_ca(z)) * ca.DM(y_sd))
            opti.subject_to(X[:, j + 1] == xn)
        opti.subject_to(opti.bounded(-V.d_max, ca.vec(U[0, :]), V.d_max)); opti.subject_to(opti.bounded(V.ax_min, ca.vec(U[1, :]), V.ax_max))
        opti.subject_to(opti.bounded(-0.5, ca.vec(X[6, :]), 0.5)); opti.subject_to(opti.bounded(1.0, ca.vec(X[3, :]), 30.0))
        opti.subject_to(opti.bounded(-1.3, ca.vec(X[2, :]), 1.3)); opti.subject_to(opti.bounded(-8.0, ca.vec(X[4, :]), 8.0)); opti.subject_to(opti.bounded(-1.5, ca.vec(X[5, :]), 1.5))
        opti.subject_to(ca.vec(S_ph) >= 0)
        if margin: opti.subject_to(ca.vec(S_m) >= 0)
        R_TOT = R_O + R_V + R_BUF
        def barriers(j, qe, qo_j, qph_j):
            px, py = X[0, j], X[1, j]; hs = []
            for m in range(M_OBS):
                hs.append(ca.sqrt((px - p_obs[2 * m, j]) ** 2 + (py - p_obs[2 * m + 1, j]) ** 2 + 1e-4) - R_TOT - qe - qo_j[m] - qph_j[m])
            qr = qe if road_margin else 0.0
            hs.append(py - (p_road[0] + R_V) - qr); hs.append((p_road[1] - R_V) - py - qr)
            return hs
        zero = [0.0] * M_OBS
        h_prev = barriers(0, 0.0, zero, zero)
        for j in range(1, n + 1):
            hp = barriers(j, 0.0, zero, [p_qph[m, j - 1] for m in range(M_OBS)])           # PHYSICAL level (obstacle inflation lives here)
            for i in range(M_OBS + 2): opti.subject_to(hp[i] >= -S_ph[i, j - 1])
            if margin:
                hj = barriers(j, p_q[j - 1], [p_qo[m, j - 1] for m in range(M_OBS)], zero)   # CBF-form soft margin level
                for i in range(M_OBS + 2):
                    if i >= M_OBS and not road_margin: continue                              # no margin on the road edges
                    opti.subject_to(hj[i] >= (1 - p_gamma) * h_prev[i] - S_m[i, j - 1])
                h_prev = hj
        cost = 0
        for j in range(1, n + 1):
            w = 3.0 if j == n else 1.0
            xj = X[0, j]; yref = PATH_A * ca.sin(_KX * xj); dyref = PATH_A * _KX * ca.cos(_KX * xj)
            psiref = ca.atan(dyref); rref = X[3, j] * (-PATH_A * _KX ** 2 * ca.sin(_KX * xj)) / (1 + dyref ** 2) ** 1.5
            cost += w * (1.0 * (X[1, j] - yref) ** 2 + 4.0 * (X[2, j] - psiref) ** 2 + 0.5 * (X[3, j] - V_REF) ** 2 + 1.0 * (X[5, j] - rref) ** 2)
        for j in range(n):
            du = U[:, j] - (p_uprev if j == 0 else U[:, j - 1])
            cost += 40.0 * U[0, j] ** 2 + 0.2 * U[1, j] ** 2 + 3000.0 * du[0] ** 2 + 1.0 * du[1] ** 2
        cost += 1e4 * ca.sum1(ca.sum2(S_ph)) + 1e3 * ca.sumsqr(S_ph)
        if margin: cost += W_M_LIN * ca.sum1(ca.sum2(S_m)) + W_M_QUAD * ca.sumsqr(S_m)
        opti.minimize(cost)
        opti.solver("ipopt", {"expand": True, "print_time": False, "ipopt.print_level": 0, "ipopt.sb": "yes", "ipopt.max_iter": 60, "ipopt.tol": 1e-4,
                              "ipopt.acceptable_tol": 1e-2, "ipopt.acceptable_iter": 3, "ipopt.mu_strategy": "adaptive"})
        self.opti, self.X, self.U, self.S_ph, self.S_m = opti, X, U, S_ph, S_m
        self.p = dict(s0=p_s0, uprev=p_uprev, obs=p_obs, q=p_q, qo=p_qo, qph=p_qph, gamma=p_gamma, th=p_th, Th=p_Th, xi=p_xi, road=p_road, tau=p_tau)
        self.reset()
    def reset(self): self.X0, self.U0 = None, None
    def solve(self, s0, uprev, obs_traj, q, gamma, qo=None, qph=None, th=None, Theta=None, xi=1.0, road=(Y_LO, Y_HI), tau=1.0):
        """obs_traj: (n+1, M_OBS, 2) predicted obstacle positions at the planning stages 0..n"""
        o, p = self.opti, self.p
        o.set_value(p["road"], np.asarray(road, float)); o.set_value(p["s0"], s0); o.set_value(p["uprev"], uprev)
        o.set_value(p["obs"], np.concatenate([obs_traj[:, m, :].T for m in range(M_OBS)], axis=0))
        o.set_value(p["q"], q); o.set_value(p["gamma"], gamma)
        o.set_value(p["qo"], np.zeros((M_OBS, self.n)) if qo is None else qo); o.set_value(p["qph"], np.zeros((M_OBS, self.n)) if qph is None else qph)
        if p["th"] is not None: o.set_value(p["th"], th)
        if p["Th"] is not None: o.set_value(p["Th"], Theta)
        if p["xi"] is not None: o.set_value(p["xi"], xi)
        if p["tau"] is not None: o.set_value(p["tau"], tau)
        if self.X0 is None: Xg = np.tile(s0[:, None], (1, self.n + 1)); Ug = np.tile(uprev[:, None], (1, self.n))
        else: Xg = np.hstack([self.X0[:, 1:], self.X0[:, -1:]]); Xg[:, 0] = s0; Ug = np.hstack([self.U0[:, 1:], self.U0[:, -1:]])
        o.set_initial(self.X, Xg); o.set_initial(self.U, Ug); o.set_initial(self.S_ph, 0)
        if self.margin: o.set_initial(self.S_m, 0)
        t0 = time.perf_counter(); ok = True
        def grab(get):
            return get(self.X), get(self.U), float(np.sum(get(self.S_ph))), (float(np.sum(get(self.S_m))) if self.margin else 0.0)
        try: sol = o.solve(); Xs, Us, sl, sm = grab(sol.value)
        except RuntimeError:
            ok = False
            try: Xs, Us, sl, sm = grab(o.debug.value)
            except Exception: Xs, Us, sl, sm = Xg, Ug, np.nan, np.nan
        dt = time.perf_counter() - t0
        Xs = np.asarray(Xs).reshape(NX, self.n + 1); Us = np.asarray(Us).reshape(NU, self.n)
        if not np.all(np.isfinite(Xs)) or not np.all(np.isfinite(Us)): Xs, Us, ok = Xg, Ug, False
        self.X0, self.U0 = Xs, Us
        return dict(u=Us[:, 0].copy(), X=Xs, U=Us, ok=ok, time=dt, slack=sl, slack_m=sm)

_MPCT = {}
def get_mpc(kind, margin, road_margin=True):
    key = (kind, bool(margin), bool(road_margin) if margin else True)
    if key not in _MPCT: _MPCT[key] = NMPCT(kind, key[1], key[2])
    return _MPCT[key]
t0 = time.time()
for _k in ("nom", "nn"):
    for _m in (False, True): get_mpc(_k, _m, True)
get_mpc("nom", True, False); get_mpc("nn", True, False)
print(f"NMPCT variants built in {time.time()-t0:.1f}s")

## P3. Moving obstacles, obstacle-position error and the obstacle-side conformal margin
**Scenario family `mo_*`** (the same road, the same ego shifts machinery; the second obstacle stays static):
| scenario | obstacle 1 (moves) | obstacle position measurement | what the controllers must cope with |
|---|---|---|---|
| `mo_cv` | constant velocity | Gaussian σ = 0.08 m | **control**: the constant-velocity prediction is right; a good margin must not hurt |
| `mo_swerve` | **abrupt lateral velocity change** (Δv = 0.9–1.5 m/s at t = 1.8–2.8 s) | σ = 0.08 m | prediction error grows with the horizon |
| `mo_stopgo` | **brakes to a stop** (0.4 s) at t = 1.8–2.8 s | σ = 0.08 m | the "ahead" prediction becomes wrong |
| `mo_noisy` | constant velocity | **heavy-tailed noise** (Student-t, 3 dof, σ = 0.25 m) | velocity estimate noise → position error at long horizons |
| `mo_swerve_noisy` | swerve | heavy-tailed noise | both |
| `mo_swerve_wet` | swerve | σ = 0.08 m | **plus** low grip (mu = 0.55): ego-model error and obstacle error together |
Optional (not run by default) `mo_bias`: a constant 0.6 m bias of the measured obstacle position — **undetectable by any score that only compares measurements with measurements**; it is the honest negative control for "perception bias".

**Perception stack (identical for all controllers):** the controller never sees the true obstacle, only noisy position measurements (inside the 32 m sensing range). A sliding-window (0.8 s) least-squares line fit gives position and velocity; the prediction is **constant velocity** over the horizon.
**Obstacle-side score (horizon `j`, obstacle `m`):** `e_j(t) = || meas_m(t) − [pos_fit_m(t−j) + vel_fit_m(t−j)·j·dt] ||` — observable only `j` steps later (the same delayed-feedback structure as the ego-model scores). It is calibrated **offline on nominally-manoeuvring obstacles** (small random accelerations, σ = 0.08 m noise), so every `mo_*` shift is out-of-distribution, and then adapted online (grouped ACI + shift propagation, `aci_prop`) or kept fixed (`static`, split-CP).
Note: because the score compares *measurements*, a constant measurement bias or a constant sensing latency cancels out of it — it measures prediction error, not perception truth. That limitation is stated, not hidden.

In [ ]:
SCEN.update({
    "mo_cv":           dict(mu=0.90, m=1500.0, mo=dict(change=None,     sigma=0.08, df=None)),
    "mo_swerve":       dict(mu=0.90, m=1500.0, mo=dict(change="swerve", sigma=0.08, df=None, t_change=(1.8, 2.8), dv=(0.9, 1.5))),
    "mo_stopgo":       dict(mu=0.90, m=1500.0, mo=dict(change="stop",   sigma=0.08, df=None, t_change=(1.8, 2.8))),
    "mo_noisy":        dict(mu=0.90, m=1500.0, mo=dict(change=None,     sigma=0.25, df=3)),
    "mo_swerve_noisy": dict(mu=0.90, m=1500.0, mo=dict(change="swerve", sigma=0.25, df=3,    t_change=(1.8, 2.8), dv=(0.9, 1.5))),
    "mo_swerve_wet":   dict(mu=0.55, m=1500.0, mo=dict(change="swerve", sigma=0.08, df=None, t_change=(1.8, 2.8), dv=(0.9, 1.5))),
    "mo_bias":         dict(mu=0.90, m=1500.0, mo=dict(change=None,     sigma=0.08, df=None, bias=0.6)),
})

def make_world2(seed, sc):
    """returns dict(OBt (T,M,2) true obstacle positions per sim step, OBm (T,M,2) measured positions or None for the old static/exact world, v0, mn, pn, gj)"""
    obs0, v0, mn, pn, gj = make_world(seed)
    n_steps = int(round(CFG["T_sim"] / DT)); T = n_steps + 2
    if "mo" not in sc:
        return dict(OBt=np.tile(obs0[None], (T, 1, 1)), OBm=None, v0=v0, mn=mn, pn=pn, gj=gj)
    mo = sc["mo"]; rng = np.random.default_rng(70_000 + seed); t = np.arange(T) * DT
    xc = obs0[0, 0]; t_arr = xc / V_REF                                        # the ego reaches obstacle 1's x at about t_arr
    vx = rng.uniform(0.0, 3.0); side = rng.choice([-1.0, 1.0]); vy = -side * rng.uniform(0.5, 1.0); d_arr = rng.uniform(0.0, 1.0)
    y_arr = path_y(xc) + side * d_arr; x0, y0 = xc - vx * t_arr, y_arr - vy * t_arr
    vxs, vys = np.full(T, vx), np.full(T, vy)
    ch = mo.get("change")
    if ch == "swerve":
        tc = rng.uniform(*mo["t_change"]); dv = rng.choice([-1.0, 1.0]) * rng.uniform(*mo["dv"]); vys = np.where(t >= tc, vy + dv, vy)
    elif ch == "stop":
        tc = rng.uniform(*mo["t_change"]); ramp = np.clip((t - tc) / 0.4, 0.0, 1.0); vxs, vys = vx * (1 - ramp), vy * (1 - ramp)
    px = x0 + np.concatenate([[0.0], np.cumsum(vxs[:-1]) * DT]); py = y0 + np.concatenate([[0.0], np.cumsum(vys[:-1]) * DT])
    OBt = np.zeros((T, M_OBS, 2)); OBt[:, 0, 0], OBt[:, 0, 1] = px, py; OBt[:, 1, :] = obs0[1][None]
    eps = rng.standard_normal((T, M_OBS, 2)) if mo.get("df") is None else rng.standard_t(mo["df"], (T, M_OBS, 2)) / math.sqrt(mo["df"] / (mo["df"] - 2))
    OBm = OBt + mo["sigma"] * eps
    if "bias" in mo: OBm[:, 0, 1] += rng.choice([-1.0, 1.0]) * mo["bias"]
    return dict(OBt=OBt, OBm=OBm, v0=v0, mn=mn, pn=pn, gj=gj)

class ObsTracker:
    """sliding-window least-squares line fit per obstacle (window W steps, velocity only after min_vel samples); history is cleared when the obstacle leaves the sensing range"""
    def __init__(self, M=M_OBS, W=8, min_vel=4):
        self.W, self.mv = W, min_vel; self.hist = [deque(maxlen=W) for _ in range(M)]
    def update(self, meas, vis):
        M = len(vis); P = np.zeros((M, 2)); Vv = np.zeros((M, 2)); ok = np.zeros(M, bool)
        for m in range(M):
            if not vis[m]: self.hist[m].clear(); continue
            self.hist[m].append(np.array(meas[m], float)); H = np.array(self.hist[m]); n = len(H)
            if n >= self.mv:
                tau = (np.arange(n) - (n - 1)) * DT; d = tau - tau.mean()
                b = (d[:, None] * (H - H.mean(0))).sum(0) / (d ** 2).sum(); P[m] = H.mean(0) - b * tau.mean(); Vv[m] = b; ok[m] = True
            else: P[m] = H[-1]
        return P, Vv, ok

def obs_score_vec(meas_k, fitP, fitV, fitOK, k, n=N):
    """e[j-1] = || meas(k) − (fitP[k-j] + fitV[k-j]·j·dt) ||  (NaN where the fit at the origin was not valid)"""
    e = np.full(n, np.nan); B = min(n, k)
    if B < 1: return e
    j = np.arange(1, B + 1); o = k - j
    err = np.linalg.norm(fitP[o] + fitV[o] * (j[:, None] * DT) - np.asarray(meas_k)[None], axis=1); err[~fitOK[o]] = np.nan
    e[:B] = err; return e

def calibrate_obs_scores(n_traj=300, sigma=0.08, a_std=0.8, T=90, seed=7):
    """in-distribution obstacle behaviour: constant velocity + small random accelerations, nominal sensor noise"""
    rng = np.random.default_rng(seed); sc_ = [[] for _ in range(N)]
    for _ in range(n_traj):
        v = np.array([rng.uniform(0, 3), rng.uniform(-1.2, 1.2)]); p = np.zeros(2); Pt = np.zeros((T, 2))
        for k in range(T):
            Pt[k] = p; v = v + a_std * DT * rng.standard_normal(2); p = p + v * DT
        Mm = Pt + sigma * rng.standard_normal(Pt.shape); trk = ObsTracker(1)
        FP, FV, FO = np.zeros((T, 2)), np.zeros((T, 2)), np.zeros(T, bool)
        for k in range(T):
            P_, V_, o_ = trk.update(Mm[k][None], np.array([True])); FP[k], FV[k], FO[k] = P_[0], V_[0], o_[0]
            e = obs_score_vec(Mm[k], FP, FV, FO, k)
            for j in range(N):
                if not np.isnan(e[j]): sc_[j].append(e[j])
    return [np.array(s) for s in sc_]
t0 = time.time(); OBS_CAL = calibrate_obs_scores()
print(f"obstacle calibration scores in {time.time()-t0:.1f}s | median / 90 % / 99 % of the position-prediction error at j=1,5,10,25 [m]:",
      [(round(float(np.median(OBS_CAL[j-1])), 2), round(float(np.quantile(OBS_CAL[j-1], .9)), 2), round(float(np.quantile(OBS_CAL[j-1], .99)), 2)) for j in (1, 5, 10, 25)])

def clearance_series2(P, OBt):
    """min clearance between the ego path P (T,2) and the TRUE (possibly moving) obstacles OBt (>=T, M, 2), with linear interpolation of both between samples"""
    T = len(P); cl = np.full(T, np.inf); al = np.linspace(0, 1, 6)
    for t in range(T - 1):
        for a in al:
            p = (1 - a) * P[t] + a * P[t + 1]; o = (1 - a) * OBt[t] + a * OBt[t + 1]
            cl[t] = min(cl[t], float(np.min(np.linalg.norm(o - p, axis=1))) - R_O - R_V)
    cl[-1] = float(np.min(np.linalg.norm(OBt[T - 1] - P[-1], axis=1))) - R_O - R_V
    return cl

## P4. Predictive CBF safety filter (baseline) and the extended closed-loop simulator
**CBF safety filter (strong baseline, two variants).** A small nonlinear program (IPOPT) that finds the *minimal* deviation `δ_0..δ_{h-1}` (h = 6 steps = 0.6 s) from the nominal controller's planned inputs such that the **discrete-time CBF conditions** `h(x_{j}) ≥ (1−γ_f) h(x_{j−1})` hold along a 12-step look-ahead of the nominal model (with the identified friction scale), for every visible obstacle (inflated by a safety radius `r_f`) and both road edges (soft slack for feasibility).
* `Adaptive-MPC + CBF filter` (**aware**): the nominal controller is the adaptive MPC that already plans around the (predicted) obstacles; the filter only corrects it.
* `Adaptive-Track + CBF filter` (**agnostic**): the nominal controller is the same MPC *without* obstacle constraints (lane tracking only) and the filter alone must keep the vehicle safe — the classical "nominal controller + CBF-QP" architecture. `r_f` is grid-tuned on the dev seeds for both.
This is a *predictive* (look-ahead) CBF filter rather than a one-step CBF-QP, i.e. a stronger variant than the textbook one; it is still not a learned CBF or a safe-RL policy (not implemented here).

In [ ]:
class CBFFilter:
    def __init__(self, H=12, hold=6):
        self.H, self.hold = H, hold
        o = ca.Opti(); D = o.variable(NU, hold); S = o.variable(M_OBS + 2, H)
        p_s0 = o.parameter(NX); p_xi = o.parameter(1); p_U = o.parameter(NU, H); p_obs = o.parameter(2 * M_OBS, H + 1)
        p_road = o.parameter(2); p_gam = o.parameter(1); p_r = o.parameter(1)
        R_TOT = R_O + R_V + R_BUF
        def bar(x, j):
            hs = [ca.sqrt((x[0] - p_obs[2 * m, j]) ** 2 + (x[1] - p_obs[2 * m + 1, j]) ** 2 + 1e-4) - R_TOT - p_r for m in range(M_OBS)]
            hs.append(x[1] - (p_road[0] + R_V)); hs.append((p_road[1] - R_V) - x[1]); return hs
        x = p_s0; hprev = bar(x, 0); th = ca.vertcat(p_xi, 0.0); cost = 0
        for j in range(H):
            u = p_U[:, j] + (D[:, j] if j < hold else ca.DM.zeros(NU))
            if j < hold:
                o.subject_to(o.bounded(-V.d_max, u[0], V.d_max)); o.subject_to(o.bounded(V.ax_min, u[1], V.ax_max))
                cost += (D[0, j] / 0.05) ** 2 + (D[1, j] / 1.0) ** 2 + (50.0 * (D[0, j] - (D[0, j - 1] if j else 0.0))) ** 2
            x = F_nom(x, u, th); hj = bar(x, j + 1)
            for i in range(M_OBS + 2): o.subject_to(hj[i] >= (1 - p_gam) * hprev[i] - S[i, j])
            hprev = hj
        o.subject_to(ca.vec(S) >= 0); cost += 1e4 * ca.sum1(ca.sum2(S)) + 1e3 * ca.sumsqr(S)
        o.minimize(cost)
        o.solver("ipopt", {"expand": True, "print_time": False, "ipopt.print_level": 0, "ipopt.sb": "yes", "ipopt.max_iter": 40, "ipopt.tol": 1e-4, "ipopt.acceptable_tol": 1e-2, "ipopt.acceptable_iter": 3})
        self.o, self.D, self.S, self.p = o, D, S, dict(s0=p_s0, xi=p_xi, U=p_U, obs=p_obs, road=p_road, gam=p_gam, r=p_r)
    def filter(self, s0, U_ref, obs_traj, xi, road, r_f, gamma_f=0.12):
        """U_ref: (NU, >=H) nominal planned inputs; obs_traj: (>=H+1, M, 2). returns (u0, ok, solve_time)"""
        o, p, H = self.o, self.p, self.H
        o.set_value(p["s0"], s0); o.set_value(p["xi"], xi); o.set_value(p["U"], U_ref[:, :H]); o.set_value(p["road"], np.asarray(road, float))
        o.set_value(p["obs"], np.concatenate([obs_traj[:H + 1, m, :].T for m in range(M_OBS)], axis=0)); o.set_value(p["gam"], gamma_f); o.set_value(p["r"], r_f)
        o.set_initial(self.D, 0); o.set_initial(self.S, 0); t0 = time.perf_counter(); ok = True
        try: d0 = np.asarray(o.solve().value(self.D)).reshape(NU, self.hold)[:, 0]
        except RuntimeError:
            ok = False
            try: d0 = np.asarray(o.debug.value(self.D)).reshape(NU, self.hold)[:, 0]
            except Exception: d0 = np.zeros(NU)
        if not np.all(np.isfinite(d0)): d0, ok = np.zeros(NU), False
        return np.array([np.clip(U_ref[0, 0] + d0[0], -V.d_max, V.d_max), np.clip(U_ref[1, 0] + d0[1], V.ax_min, V.ax_max)]), ok, time.perf_counter() - t0
CBF = CBFFilter()

In [ ]:
def simulate2(name, spec, scen_name, seed, keep=False, delay=None):
    """Extended simulator. New spec keys (all optional, defaults reproduce `simulate`):
       road_margin (bool) · q_cap · obs ('cv' | 'oracle') · infl (m/s, tuned velocity-uncertainty inflation at the physical level) · obs_tube (None|'static'|'aci_prop') · obs_level ('phys'|'soft') ·
       cbf (None|'aware'|'agnostic') · cbf_r (safety radius of the filter [m])  · cbf_gamma (decay rate of the filter barrier, default 0.12) · obs_scale (post-hoc control: multiplies the obstacle-motion margin, capped at OBS_QCAP; default 1 = unchanged)"""
    delay = DELAY if delay is None else delay
    sc = SCEN[scen_name]; Wd = make_world2(seed, sc); OBt, OBm = Wd["OBt"], Wd["OBm"]; v0, mn, pn, gj = Wd["v0"], Wd["mn"], Wd["pn"], Wd["gj"]
    moving = OBm is not None
    plant = make_plant(sc); s = plant.reset(v0)
    ylo, yhi, rsense = sc.get("ylo", Y_LO), sc.get("yhi", Y_HI), sc.get("r_sense", R_SENSE)
    bias = np.zeros(NX)
    for i_, v_ in sc.get("bias", {}).items(): bias[i_] = v_
    kind = spec["model"]; use_tube = spec["tube"] is not None; mpc = get_mpc(kind, use_tube, spec.get("road_margin", True)); mpc.reset()
    q_cap = spec.get("q_cap", Q_CAP); use_nn = kind == "nn"; Theta = THETA0; xi = 1.0; tau = 1.0
    gate = ResidualGate(THETA0) if (use_nn and spec.get("gate")) else None
    est = XiEstimator(THETA0 if use_nn else None) if spec["adapt"] == "rls" else (XiMHE(THETA0 if use_nn else None) if spec["adapt"] == "mhe" else None)
    tube = make_tube(spec, use_nn, seed) if spec["tube"] in ("static", "const", "aci", "aci_prop", "aci_joint") else None
    tracker = ObsTracker() if moving else None
    otubes = [ConformalTube(OBS_CAL, mode=spec["obs_tube"], alpha=spec.get("obs_alpha", ALPHA), eta=spec.get("obs_eta", ETA), q_cap=OBS_QCAP, seed=seed + 17 * m) for m in range(M_OBS)] if (moving and spec.get("obs_tube")) else None
    cbf_mode = spec.get("cbf"); obs_mode = spec.get("obs", "cv")
    n_steps = int(round(CFG["T_sim"] / DT)); T1 = n_steps + 2
    FP, FV = np.zeros((T1, M_OBS, 2)), np.zeros((T1, M_OBS, 2)); FOK, VIS = np.zeros((T1, M_OBS), bool), np.zeros((T1, M_OBS), bool)
    Sm, Uh, Str = [], [], [s.copy()]; u_pending = np.zeros(2)
    q_prev = np.zeros(N); qo_prev = np.zeros((M_OBS, N))
    q_log, qo_log, cov_log, covo_log, ok_log, t_log, slack_log, slackm_log, xi_log, tau_log, cbfok_log = [], [], [], [], [], [], [], [], [], [], []
    terminated = finished = False; far = np.array([0.0, 100.0])
    for k in range(n_steps):
        sm = s + bias + mn[k]; Sm.append(sm)
        src = OBm[k] if moving else OBt[k]
        vis = ((src[:, 0] - sm[0]) < rsense) & ((src[:, 0] - sm[0]) > -5.0); VIS[k] = vis
        if moving:
            P_, V_, o_ = tracker.update(OBm[k], vis); FP[k], FV[k], FOK[k] = P_, V_, o_
        if k >= 1:
            if est is not None: est.update(Sm[k - 1], Uh[k - 1], sm, tau); xi = est.xi
            if gate is not None: tau = gate.update(Sm, Uh, k, xi)
            if use_tube:
                e = multistep_scores(Sm, Uh, k, Theta if use_nn else None, use_nn=use_nn, xi=xi, tau=tau)
                cov_log.append(np.where(np.isnan(e), np.nan, (e <= q_prev + 1e-9).astype(float)))
                if tube is not None: tube.update(e)
            if otubes is not None:
                for m in range(M_OBS):
                    if not VIS[k, m]: continue
                    eo = obs_score_vec(OBm[k, m], FP[:, m], FV[:, m], FOK[:, m], k)
                    if m == 0: covo_log.append(np.where(np.isnan(eo), np.nan, (eo <= qo_prev[m] + 1e-9).astype(float)))
                    otubes[m].update(eo)
        # ---- margins
        if not use_tube: q = np.zeros(N)
        elif spec["tube"] == "ens": q = ensemble_margins(sm, mpc.U0 if mpc.U0 is not None else np.zeros((NU, N)), xi=xi, tau=tau)
        else: q = tube.margins()
        q = np.clip(np.nan_to_num(q, nan=q_cap, posinf=q_cap), 0.0, q_cap); q_prev = q.copy()
        sh = (lambda a: np.concatenate([a[..., 1:], a[..., -1:]], axis=-1)) if delay else (lambda a: a)
        q_plan = sh(q)
        qo = np.zeros((M_OBS, N))
        if otubes is not None:
            for m in range(M_OBS):
                if VIS[k, m]: qo[m] = np.clip(spec.get("obs_scale", 1.0) * np.nan_to_num(otubes[m].margins(), nan=OBS_QCAP, posinf=OBS_QCAP), 0.0, OBS_QCAP)
        qo_prev = qo.copy(); qo_plan = sh(qo)
        qph = np.zeros((M_OBS, N)); qo_soft = np.zeros((M_OBS, N))
        if spec.get("infl", 0.0) > 0: qph += spec["infl"] * ((delay + 1 + np.arange(N)) * DT)[None, :]
        if otubes is not None:
            if spec.get("obs_level", "phys") == "phys": qph = qph + qo_plan
            else: qo_soft = qo_plan
        # ---- obstacle trajectory over the planning stages (stage j <-> time (k + delay + j) dt)
        tt = (delay + np.arange(N + 1)) * DT; obs_traj = np.zeros((N + 1, M_OBS, 2))
        for m in range(M_OBS):
            if not vis[m]: obs_traj[:, m] = [sm[0] + 500.0, 100.0]
            elif not moving: obs_traj[:, m] = OBt[0, m]
            elif obs_mode == "oracle": obs_traj[:, m] = OBt[np.minimum(k + delay + np.arange(N + 1), T1 - 1), m]
            else: obs_traj[:, m] = FP[k, m][None] + FV[k, m][None] * tt[:, None]
        obs_mpc = obs_traj.copy()
        if cbf_mode == "agnostic": obs_mpc[:] = [sm[0] + 500.0, 100.0]
        # ---- solve (possibly from the model-predicted state when there is a computational delay)
        if delay == 0: s_plan, u_prev_plan, t_plan = sm, (Uh[-1] if Uh else np.zeros(2)), k * DT; u_cur = None
        else: u_cur = u_pending; s_plan = predict_next(sm, u_cur, kind, Theta, xi, sc, k * DT, tau); u_prev_plan, t_plan = u_cur, (k + 1) * DT
        res = mpc.solve(s_plan, u_prev_plan, obs_mpc, q_plan, spec["gamma"], qo=qo_soft, qph=qph, th=(oracle_th(sc, t_plan) if kind == "oracle" else None), Theta=Theta, xi=xi, road=(ylo, yhi), tau=tau)
        u_new = np.array([np.clip(res["u"][0], -V.d_max, V.d_max), np.clip(res["u"][1], V.ax_min, V.ax_max)]); tsolve = res["time"]
        if cbf_mode:
            u_new, okc, tc_ = CBF.filter(s_plan, res["U"], obs_traj, xi, (ylo, yhi), spec.get("cbf_r", 0.3), spec.get("cbf_gamma", 0.12)); tsolve += tc_; cbfok_log.append(okc)
        if delay == 0: u_cur = u_new
        else: u_pending = u_new
        xi_log.append(xi); tau_log.append(tau)
        t_log.append(tsolve); ok_log.append(res["ok"]); slack_log.append(res["slack"]); slackm_log.append(res["slack_m"]); q_log.append(q); qo_log.append(qo[0])
        # ---- plant
        gust = sc["gust_a"] * DT if ("gust_t0" in sc and sc["gust_t0"] + gj <= k * DT < sc["gust_t0"] + gj + sc["gust_dur"]) else 0.0
        s = plant.step(u_cur, k * DT, pn[k], gust); Uh.append(u_cur); Str.append(s.copy())
        if abs(s[2]) > 1.2 or s[3] < 1.5 or abs(s[4]) > 6.0 or not np.all(np.isfinite(s)): terminated = True; break
        if s[0] >= X_END: finished = True; break
    Str = np.array(Str); P = Str[:, :2]; Uh_a = np.array(Uh)
    cl = clearance_series2(P, OBt); min_clear = float(cl.min())
    road_viol = float(max(0.0, np.max((ylo + R_V) - P[:, 1]), np.max(P[:, 1] - (yhi - R_V))))
    collision = min_clear < 0; road = road_viol > 0.05
    cov = np.array(cov_log) if cov_log else np.zeros((0, N)); covo = np.array(covo_log) if covo_log else np.zeros((0, N))
    cov_g = [float(np.nanmean(cov[:, a - 1:b])) if cov.size and np.any(~np.isnan(cov[:, a - 1:b])) else np.nan for (a, b) in GROUPS_REPORT]
    covo_g = [float(np.nanmean(covo[:, a - 1:b])) if covo.size and np.any(~np.isnan(covo[:, a - 1:b])) else np.nan for (a, b) in GROUPS_REPORT]
    qlog = np.array(q_log); qolog = np.array(qo_log); vis0 = VIS[:len(qolog), 0]
    ay = Str[:, 3] * Str[:, 5]; jerk = np.diff(Str[:, 7]) / DT; reach = np.where(P[:, 0] >= 95.0)[0]
    m_ = dict(controller=name, scenario=scen_name, seed=seed, success=float((not collision) and (not road) and (not terminated)), collision=float(collision), road_violation=float(road),
              terminated=float(terminated), finished=float(finished), min_clearance=min_clear, road_viol_m=road_viol, rms_y=float(np.sqrt(np.mean((P[:, 1] - path_y(P[:, 0])) ** 2))),
              mean_speed=float(Str[:, 3].mean()), progress=float(P[-1, 0]), t95=float(reach[0] * DT) if len(reach) else np.nan, ay_rms=float(np.sqrt(np.mean(ay ** 2))), jerk_rms=float(np.sqrt(np.mean(jerk ** 2))),
              solve_ms=1e3 * float(np.mean(t_log)), solve_ms_p95=1e3 * float(np.quantile(t_log, 0.95)), deadline_miss=float(np.mean(np.array(t_log) > DT)),
              solver_fail=float(1 - np.mean(ok_log)), cbf_fail=float(1 - np.mean(cbfok_log)) if cbfok_log else np.nan, slack_mean=float(np.nanmean(slack_log)), slack_margin=float(np.nanmean(slackm_log)),
              q_mid=float(qlog[:, 11].mean()), q_end=float(qlog[:, -1].mean()), tau_mean=float(np.mean(tau_log)), xi_end=float(xi_log[-1]),
              qo_mid=float(qolog[vis0, 9].mean()) if vis0.any() else np.nan, qo_end=float(qolog[vis0, -1].mean()) if vis0.any() else np.nan,
              cov_overall=float(np.nanmean(cov)) if cov.size and np.any(~np.isnan(cov)) else np.nan, cov_g1=cov_g[0], cov_g2=cov_g[1], cov_g3=cov_g[2],
              covo_overall=float(np.nanmean(covo)) if covo.size and np.any(~np.isnan(covo)) else np.nan, covo_g1=covo_g[0], covo_g2=covo_g[1], covo_g3=covo_g[2])
    ex = dict(Str=Str, Uh=Uh_a, OBt=OBt, OBm=OBm, FP=FP, FV=FV, qlog=qlog, qolog=qolog, xi=np.array(xi_log), tau=np.array(tau_log), slack=np.array(slack_log), slackm=np.array(slackm_log), VIS=VIS) if keep else None
    return m_, ex
print("simulate2 ready")

### P4.1 Regression test: `simulate2` must reproduce the validated `simulate` on the old static scenarios
(Different NLP variable ordering → tiny numerical differences are expected; large differences would mean a wiring bug.)

In [ ]:
def regression_check():
    cases = [("Adaptive-MPC (FricID)", "ice", 1000), ("CA-NMPC (ours)", "mass_cog", 1000)]
    rows = []
    for nm, scn, sd in cases:
        a, _ = simulate(nm, SPECS[nm], scn, sd); b, _ = simulate2(nm, SPECS[nm], scn, sd)
        rows.append(dict(case=f"{nm} | {scn}", succ_old=a["success"], succ_new=b["success"], minclr_old=round(a["min_clearance"], 3), minclr_new=round(b["min_clearance"], 3), speed_old=round(a["mean_speed"], 3), speed_new=round(b["mean_speed"], 3)))
    d = pd.DataFrame(rows); print(d.to_string(index=False))
    ok = bool(((d.succ_old == d.succ_new) & ((d.minclr_old - d.minclr_new).abs() < 0.25) & ((d.speed_old - d.speed_new).abs() < 0.3)).all())
    print("REGRESSION", "OK ✔" if ok else "MISMATCH ✘ -- do not trust the pilot results until this is understood"); return ok
REG_OK = regression_check()

## P5. Utilities: resumable runner, Wilson / bootstrap / exact-McNemar / Holm

In [ ]:
import multiprocessing as mp
from scipy import stats
N_JOBS = int(os.environ.get("CA_JOBS", min(2, os.cpu_count() or 1)))
CKPT2 = f"{PO}/episodes.pkl"

def _job2(a):
    name, spec, sc, seed, delay, label = a
    try:
        m, _ = simulate2(name, spec, sc, seed, delay=delay)
    except Exception as ex_:                       # never lose a multi-hour run because of one numerical failure: record it as a failed episode
        import traceback; print("CRASH", name, sc, seed, repr(ex_)); traceback.print_exc()
        m = dict(controller=name, scenario=sc, seed=seed, success=0.0, collision=0.0, road_violation=0.0, terminated=1.0, min_clearance=np.nan, road_viol_m=0.0, crashed=1.0)
    m["label"] = label; m["delay"] = DELAY if delay is None else delay
    return m

def load_ckpt(path):
    rows = []
    if path and os.path.exists(path):
        with open(path, "rb") as f:
            while True:
                try: rows.append(pickle.load(f))
                except (EOFError, pickle.UnpicklingError): break
    return rows

def run_jobs2(jobs, ckpt=None, n_jobs=N_JOBS):
    """jobs: list of (name, spec, scenario, seed, delay, label). Resumable: finished episodes are appended to `ckpt` and skipped on re-run."""
    key = lambda r: (r.get("label"), r["controller"], r["scenario"], r["seed"], r.get("delay"))
    done = {key(r): r for r in load_ckpt(ckpt)}
    todo = [j for j in jobs if (j[5], j[0], j[2], j[3], DELAY if j[4] is None else j[4]) not in done]
    t0 = time.time(); new = []
    print(f"  {len(jobs) - len(todo)} episodes found in checkpoint, {len(todo)} to run", flush=True)
    fh = open(ckpt, "ab") if ckpt else None
    def sink(m):
        new.append(m)
        if fh: pickle.dump(m, fh); fh.flush()
    if n_jobs > 1 and todo:
        torch.set_num_threads(1)
        with mp.get_context("fork").Pool(n_jobs) as pool:
            for i, m in enumerate(pool.imap_unordered(_job2, todo, chunksize=1), 1):
                sink(m)
                if i % 20 == 0 or i == len(todo): print(f"  {i}/{len(todo)} episodes | {time.time()-t0:.0f}s", flush=True)
    else:
        for i, a in enumerate(todo, 1):
            sink(_job2(a))
            if i % 10 == 0 or i == len(todo): print(f"  {i}/{len(todo)} episodes | {time.time()-t0:.0f}s", flush=True)
    if fh: fh.close()
    want = {(j[5], j[0], j[2], j[3], DELAY if j[4] is None else j[4]) for j in jobs}
    allr = {key(r): r for r in list(done.values()) + new}
    out = pd.DataFrame([allr[k] for k in want if k in allr])
    if "finished" in out: out["success"] = out["success"] * out["finished"].fillna(0)      # an episode only counts as a success if the vehicle also REACHES x = 100 m (a crawling controller must not "succeed")
    return out

def make_jobs(specs, names, scenarios, seeds, label):
    return [(n, specs[n], s, sd, None, label) for sd in seeds for s in scenarios for n in names]

def violation_of(d):
    return np.maximum(0.0, -d["min_clearance"].fillna(0)) + d["road_viol_m"].fillna(0) + 10.0 * d["terminated"].fillna(1)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))
def holm(pvals):
    p = np.array(pvals, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order): run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj
def boot_ci(x, B=4000, seed=0):
    x = np.asarray(x, float)
    if len(x) == 0: return (np.nan, np.nan)
    r = np.random.default_rng(seed); idx = r.integers(0, len(x), (B, len(x))); return tuple(np.quantile(x[idx].mean(1), [0.025, 0.975]))
def cohen_h(p1, p2): return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

def paired_vs(d, ref, others, scen_list):
    """paired by (scenario, seed). positive difference = `ref` better. exact two-sided McNemar on success, paired bootstrap CI of the success difference, Cohen's h, Wilcoxon on the violation scalar. Holm over `others`."""
    x = d[d.scenario.isin(scen_list)].copy(); x["violation_m"] = violation_of(x)
    piv = x.pivot_table(index=["scenario", "seed"], columns="controller", values=["success", "violation_m"], observed=False)
    rows = []
    for c in others:
        if c == ref or c not in piv["success"].columns or ref not in piv["success"].columns: continue
        a, b = piv["success"][ref].values, piv["success"][c].values; ok = ~(np.isnan(a) | np.isnan(b)); a, b = a[ok], b[ok]
        n01 = int(((a == 1) & (b == 0)).sum()); n10 = int(((a == 0) & (b == 1)).sum())
        p_mc = stats.binomtest(n01, n01 + n10, 0.5, alternative="two-sided").pvalue if (n01 + n10) > 0 else 1.0
        va, vb = piv["violation_m"][ref].values[ok], piv["violation_m"][c].values[ok]
        try: p_w = stats.wilcoxon(vb - va, alternative="two-sided").pvalue if np.any(vb != va) else 1.0
        except ValueError: p_w = 1.0
        lo, hi = boot_ci(a - b)
        rows.append(dict(vs=c, n_pairs=int(ok.sum()), ref_only=n01, other_only=n10, diff_pp=100 * float((a - b).mean()), ci_lo_pp=100 * lo, ci_hi_pp=100 * hi,
                         cohen_h=float(cohen_h(a.mean(), b.mean())), p_McNemar=p_mc, viol_diff_m=float((vb - va).mean()), p_Wilcoxon=p_w))
    t = pd.DataFrame(rows).set_index("vs") if rows else pd.DataFrame()
    if len(t): t["p_McNemar_holm"] = holm(t.p_McNemar.values); t["p_Wilcoxon_holm"] = holm(t.p_Wilcoxon.values)
    return t

def success_table(d, order=None):
    rows = []
    for c in (order or list(d.controller.unique())):
        x = d[d.controller == c]
        if x.empty: continue
        k, n = int(x.success.sum()), len(x); lo, hi = wilson(k, n)
        rows.append(dict(Controller=c, n=n, **{"Success %": 100 * k / n, "CI lo": 100 * lo, "CI hi": 100 * hi, "Collision %": 100 * x.collision.mean(), "Road-dep. %": 100 * x.road_violation.mean(),
                    "Spin-out %": 100 * x.terminated.mean(), "Safe % (no finish req.)": 100 * ((1 - x.collision) * (1 - x.road_violation) * (1 - x.terminated)).mean(), "Finished %": 100 * x.finished.mean(), "Min clear. m": x.min_clearance.mean(), "Speed m/s": x.mean_speed.mean(), "t95 s": x.t95.mean(), "Jerk rms": x.jerk_rms.mean(),
                    "Solve ms": x.solve_ms.mean(), "p95 ms": x.solve_ms_p95.mean(), "Deadline miss": x.deadline_miss.mean()}))
    return pd.DataFrame(rows).set_index("Controller")

PAL = {"E0": "#555555", "base": "#555555"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6, "font.size": 9,
                     "axes.titlesize": 10, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
print("utilities ready")

## P6. STEP 1 — Why does CA-NMPC fail in `tire_stiff`?  (diagnosis on the *diag* seeds; test seeds are not touched)
**What the earlier benchmark showed (multi-body plant, 3 test seeds):** in `tire_stiff` (cornering stiffness −75 %) `Adaptive-MPC (FricID)` succeeded 3/3 while `CA-NMPC` failed 0/3, `Adaptive+ACI (no NN)` 1/3 and `Nominal+ACI` 0/3 — i.e. *every controller with a tube* was worse than the same controller without a tube, while the fixed-margin tubes were fine. That points at the **dynamic margin** (or at how margins are applied), not at the neural residual.
**Candidate causes and the intervention that isolates each:**
| code | intervention | isolates |
|---|---|---|
| E1a | tube machinery on, `q ≡ 0`, `γ = 1` | pure wiring/bug: must equal E0 |
| E1b | tube machinery on, `q ≡ 0`, `γ = 0.5` | the CBF-form decay constraint `h_j ≥ (1−γ) h_{j−1}` *itself* (without any margin) |
| E2 / E5 | Adaptive+ACI / CA-NMPC defaults | the failure to explain |
| E2h | E2 with `γ = 1` (hard tightened constraint) | CBF-form |
| E3, E6 | `q_cap = 1 m` instead of 3 m | margin magnitude |
| E4, E7 | **no ego-tube margin on the road edges** (obstacles only) | the margins on both road edges (the free corridor is only 5.2 m wide: a 3 m margin on each side is infeasible by construction) |
| E8 | E7 + `q_cap = 1 m` | both |
| E9 | NN + gate, no tube | the neural residual alone |
**Protocol:** `tire_stiff`, diag seeds; the verdict below is automatic and prints its evidence; the *fix* (the best CA-NMPC variant among E5-E8) is chosen on the diag seeds and then **frozen** for Step 2 (test seeds).

In [ ]:
S1 = {
    "E0 Adaptive-MPC":                    dict(model="nom", adapt="rls", tube=None,       gamma=1.0),
    "E1a Adaptive + null tube (g=1)":     dict(model="nom", adapt="rls", tube="const",    gamma=1.0,   c=0.0),
    "E1b Adaptive + null tube (g=0.5)":   dict(model="nom", adapt="rls", tube="const",    gamma=GAMMA, c=0.0),
    "E2 Adaptive+ACI (default)":          dict(model="nom", adapt="rls", tube="aci_prop", gamma=GAMMA),
    "E2h Adaptive+ACI, hard (g=1)":       dict(model="nom", adapt="rls", tube="aci_prop", gamma=1.0),
    "E3 Adaptive+ACI, q_cap=1":           dict(model="nom", adapt="rls", tube="aci_prop", gamma=GAMMA, q_cap=1.0),
    "E4 Adaptive+ACI, no road margin":    dict(model="nom", adapt="rls", tube="aci_prop", gamma=GAMMA, road_margin=False),
    "E5 CA-NMPC (default)":               dict(model="nn",  adapt="rls", tube="aci_prop", gamma=GAMMA, gate=True),
    "E6 CA-NMPC, q_cap=1":                dict(model="nn",  adapt="rls", tube="aci_prop", gamma=GAMMA, gate=True, q_cap=1.0),
    "E7 CA-NMPC, no road margin":         dict(model="nn",  adapt="rls", tube="aci_prop", gamma=GAMMA, gate=True, road_margin=False),
    "E8 CA-NMPC, no road margin + q_cap=1": dict(model="nn", adapt="rls", tube="aci_prop", gamma=GAMMA, gate=True, road_margin=False, q_cap=1.0),
    "E9 NN+FricID+Gate (no tube)":        dict(model="nn",  adapt="rls", tube=None,       gamma=1.0,   gate=True),
}
t0 = time.time()
df1 = run_jobs2(make_jobs(S1, list(S1), ["tire_stiff"], PC["diag"], "step1"), ckpt=CKPT2)
df1["violation_m"] = violation_of(df1); df1.drop(columns=[c for c in df1.columns if c.startswith("cov_j")], errors="ignore").to_csv(f"{PO}/step1_episodes.csv", index=False)
print(f"Step 1: {len(df1)} episodes ({time.time()-t0:.0f}s this session)")

In [ ]:
T1 = success_table(df1, list(S1))
agg = df1.groupby("controller").agg(q_mid=("q_mid", "mean"), q_end=("q_end", "mean"), slack_margin=("slack_margin", "mean"), tau=("tau_mean", "mean"), xi_end=("xi_end", "mean"), solver_fail=("solver_fail", "mean"))
T1 = T1.join(agg).loc[list(S1)]
print(f"=== STEP 1: tire_stiff on the diag seeds (n = {len(PC['diag'])} per controller; plant = {PLANT}) ===")
print(T1[["n", "Success %", "CI lo", "CI hi", "Collision %", "Road-dep. %", "Spin-out %", "Min clear. m", "q_mid", "q_end", "slack_margin", "tau", "xi_end", "solver_fail"]].round(2).to_string())
T1.round(3).to_csv(f"{PO}/step1_table.csv")
pw1 = paired_vs(df1, "E5 CA-NMPC (default)", [k for k in S1 if k != "E5 CA-NMPC (default)"], ["tire_stiff"]); print("\npaired vs E5 (positive = E5 better; exact McNemar, n small → low power):"); print(pw1[["n_pairs", "ref_only", "other_only", "diff_pp", "p_McNemar"]].round(3).to_string())

In [ ]:
def diagnose_step1(d):
    S = d.groupby("controller").success.mean(); g = lambda k: float(S[[c for c in S.index if c.startswith(k + " ")][0]])
    ev, base = [], g("E0")
    ev.append(f"E0 Adaptive-MPC success = {base:.2f}")
    if abs(g("E1a") - base) > 0.15: ev.append(f"WIRING?  E1a (tube machinery on, q=0, gamma=1) = {g('E1a'):.2f} differs from E0 = {base:.2f} by > 0.15 → suspect an implementation problem")
    else: ev.append(f"wiring OK: E1a = {g('E1a'):.2f} ≈ E0")
    if g("E1b") < g("E1a") - 0.2: ev.append(f"CBF-FORM: E1b (q=0, gamma=0.5) = {g('E1b'):.2f} < E1a = {g('E1a'):.2f} → the decay constraint itself hurts, even without any margin")
    for branch, d0, cands in (("Adaptive+ACI", "E2", {"road-edge margin": "E4", "q_cap=1": "E3", "CBF-form (gamma=1)": "E2h"}), ("CA-NMPC", "E5", {"road-edge margin": "E7", "q_cap=1": "E6", "road-edge margin + q_cap=1": "E8"})):
        s0 = g(d0); ev.append(f"{branch}: default {d0} = {s0:.2f} ({'WORSE' if s0 < base - 0.2 else 'not clearly worse'} than E0)")
        gains = {k: g(v) - s0 for k, v in cands.items()}; best = max(gains, key=gains.get)
        ev.append(f"   gains over default: " + ", ".join(f"{k}: {v:+.2f}" for k, v in gains.items()))
        if s0 < base - 0.2 and gains[best] >= 0.2: ev.append(f"   → MAIN CAUSE in the {branch} branch: {best}")
        elif s0 < base - 0.2: ev.append(f"   → NOT explained by the tested interventions (genuine weakness or another cause) — inspect the traces")
    ev.append(f"NN alone: E9 = {g('E9'):.2f}")
    return ev
for line in diagnose_step1(df1): print(line)

In [ ]:
cands = [k for k in S1 if k.startswith(("E5", "E6", "E7", "E8"))]
Sx = df1[df1.controller.isin(cands)].groupby("controller").success.mean().reindex(cands)
best = Sx.idxmax(); FIX = dict(road_margin=bool(S1[best].get("road_margin", True)), q_cap=float(S1[best].get("q_cap", Q_CAP)), source=best, diag_success=float(Sx.max()), default_success=float(Sx.iloc[0]))
json.dump(FIX, open(f"{PO}/step1_fix.json", "w"), indent=1)
print("FIX chosen on the diag seeds and FROZEN for Step 2:", FIX)
if FIX["source"].startswith("E5"): print("(the default CA-NMPC was the best of E5-E8 → no change of the method is justified by Step 1)")

fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
yy = np.arange(len(S1))[::-1]
for i, c in enumerate(S1):
    r = T1.loc[c]; col = "#D55E00" if c.startswith(("E5", "E6", "E7", "E8")) else ("#0072B2" if c.startswith(("E2", "E3", "E4")) else "#777777")
    axs[0].plot([r["CI lo"], r["CI hi"]], [yy[i]] * 2, color=col, lw=2); axs[0].plot(r["Success %"], yy[i], "o", color=col)
axs[0].set_yticks(yy); axs[0].set_yticklabels(list(S1), fontsize=7.5); axs[0].set_xlabel("success % (Wilson 95 %)"); axs[0].set_title("tire_stiff — diag seeds")
ex_seed = PC["diag"][0]; show = ["E0 Adaptive-MPC", "E2 Adaptive+ACI (default)", "E5 CA-NMPC (default)", best]
for c, col in zip(dict.fromkeys(show), ["#555555", "#0072B2", "#D55E00", "#009E73"]):
    m_, ex_ = simulate2(c, S1[c], "tire_stiff", ex_seed, keep=True)
    axs[1].plot(ex_["Str"][:, 0], ex_["Str"][:, 1], color=col, lw=1.6, label=f"{c.split(' ')[0]} ({'ok' if m_['success'] else 'FAIL'})")
    axs[2].plot(np.arange(len(ex_["qlog"])) * DT, ex_["qlog"][:, -1], color=col, lw=1.4, label=c.split(" ")[0])
obs_ = make_world2(ex_seed, SCEN["tire_stiff"])["OBt"][0]
for o_ in obs_: axs[1].add_patch(plt.Circle(o_, R_O, color="k", alpha=0.25))
axs[1].axhline(Y_LO, color="k", lw=0.8); axs[1].axhline(Y_HI, color="k", lw=0.8); axs[1].set_xlabel("x [m]"); axs[1].set_ylabel("y [m]"); axs[1].set_title(f"paths (seed {ex_seed})"); axs[1].legend(fontsize=7)
axs[2].set_xlabel("t [s]"); axs[2].set_ylabel("tube margin at j = 25 [m]"); axs[2].set_title("margin growth"); axs[2].legend(fontsize=7)
fig.savefig(f"{PO}/fig_step1_tire_stiff.png"); plt.show()

## P7. STEP 2 — moving obstacles and obstacle-position error: controllers
| name | ego model | obstacle handling | role |
|---|---|---|---|
| `Adaptive-MPC (FricID)` | nominal + RLS friction | constant-velocity prediction, hard physical constraint | **strong classical baseline** |
| `Adaptive+TunedInflation` | same | + **velocity-uncertainty inflation** `r_j = c·j·dt` at the physical level, `c` grid-tuned on dev seeds | **robust/tube-MPC-like baseline** |
| `Adaptive-MPC + CBF filter` | same | the above MPC corrected by a **predictive CBF safety filter** (`gamma_f` and `r_f` jointly tuned on dev seeds) | CBF baseline (aware nominal layer) |
| `Adaptive-Track + CBF filter` | same | lane-tracking MPC **without** obstacles + the CBF filter does all the avoidance | CBF baseline (classical architecture) |
| `Adaptive+ObsStaticCP` | same | **offline split-conformal** obstacle margin (physical level), never updated | calibrated but not adaptive |
| `Adaptive+ObsACI` | same | **online grouped-ACI** obstacle margin with delayed feedback + shift propagation (physical level) | obstacle-side conformal, *no NN, no ego tube* |
| `CA-NMPC (ego tube)` | nominal + NN residual + gate + RLS | ego-model ACI tube only (soft CBF level; Step 1 fix applied) | the earlier method on the new scenarios |
| **`CA-NMPC-Obs (ours)`** | same | ego-model ACI tube **+** obstacle-side ACI margin | **proposed method** |
| `Adaptive + true obstacle future (ref.)` | same | the true future obstacle motion is given | reference upper bound on what *perception prediction* costs (not a competitor) |
**Parameters that are tuned (on dev seeds, never on test seeds):** `c` of the inflation baseline and `(gamma_f, r_f)` of the CBF filter (in the first quick run only `r_f` was tuned with `gamma_f` fixed at 0.12; that filter braked to ~9 m/s and never reached the goal, so it was re-tuned jointly). Everything of our own method is fixed *a priori* (`alpha = 0.1`, `eta = 0.05`, `q_cap_obs = 3 m`, `gamma = 0.5`).

In [ ]:
FIX = json.load(open(f"{PO}/step1_fix.json")) if os.path.exists(f"{PO}/step1_fix.json") else dict(road_margin=True, q_cap=Q_CAP, source="default")
print("Step-1 fix applied to the CA-NMPC controllers of Step 2:", FIX)
_base = dict(model="nom", adapt="rls", tube=None, gamma=1.0, obs="cv")
_ca = dict(model="nn", adapt="rls", tube="aci_prop", gamma=GAMMA, gate=True, obs="cv", road_margin=FIX["road_margin"], q_cap=FIX["q_cap"])
PSPECS = {
    "Adaptive-MPC (FricID)":                    dict(_base),
    "Adaptive+TunedInflation":                  dict(_base, infl=0.3),
    "Adaptive-MPC + CBF filter":                dict(_base, cbf="aware",    cbf_r=0.3, cbf_gamma=0.3),
    "Adaptive-Track + CBF filter":              dict(_base, cbf="agnostic", cbf_r=0.3, cbf_gamma=0.3),
    "Adaptive+ObsStaticCP":                     dict(_base, obs_tube="static",   obs_level="phys"),
    "Adaptive+ObsACI":                          dict(_base, obs_tube="aci_prop", obs_level="phys"),
    "CA-NMPC (ego tube)":                       dict(_ca),
    "CA-NMPC-Obs (ours)":                       dict(_ca, obs_tube="aci_prop", obs_level="phys"),
    "Adaptive + true obstacle future (ref.)":   dict(_base, obs="oracle"),
}
PRIMARY, SECONDARY = "CA-NMPC-Obs (ours)", "Adaptive+ObsACI"
B4 = ["Adaptive-MPC (FricID)", "Adaptive+TunedInflation", "Adaptive-MPC + CBF filter", "Adaptive-Track + CBF filter"]
SCN2 = list(PC["scen"]); SHIFT2 = [s for s in SCN2 if s != "mo_cv"]
PCOL = {"Adaptive-MPC (FricID)": "#555555", "Adaptive+TunedInflation": "#B2A400", "Adaptive-MPC + CBF filter": "#6A3D9A", "Adaptive-Track + CBF filter": "#CC79A7", "Adaptive+ObsStaticCP": "#56B4E9",
        "Adaptive+ObsACI": "#009E73", "CA-NMPC (ego tube)": "#E69F00", "CA-NMPC-Obs (ours)": "#D55E00", "Adaptive + true obstacle future (ref.)": "#222222"}

### P7.1 Tuning of the baselines' free parameters on the *dev* seeds + scenario-difficulty gate
The scenario severities in `SCEN` (swerve Δv, stop time, noise level) were **written without being tuned** (this notebook was assembled without being executed). The gate below therefore checks, on the dev seeds and with `Adaptive-MPC` only, that the `mo_*` scenarios are neither trivial (≥ 95 % success everywhere) nor impossible (≤ 5 %), and that the control scenario `mo_cv` is solvable.
If the gate fires, **adjust the severity in `SCEN` (dev seeds only — never look at test seeds), delete the `pilot_*` folder and re-run**.

In [ ]:
def tune_param(name, key, grid, scen, seeds):
    jobs = []
    for g in grid: jobs += make_jobs({name: dict(PSPECS[name], **{key: g})}, [name], scen, seeds, f"tune_{name}_{key}={g}")
    d = run_jobs2(jobs, ckpt=CKPT2); d["violation_m"] = violation_of(d)
    tab = d.groupby("label").agg(success=("success", "mean"), violation=("violation_m", "mean"), speed=("mean_speed", "mean")).reset_index()
    tab[key] = tab.label.str.extract(rf"{key}=(.*)$")[0].astype(float)
    tab = tab.sort_values(["success", "violation", key], ascending=[False, True, True]).reset_index(drop=True)
    print(f"--- tuning {key} of {name} on dev seeds {seeds} over {scen} ---"); print(tab[[key, "success", "violation", "speed"]].round(3).to_string(index=False))
    return float(tab.loc[0, key]), d

def tune_grid(name, grid, scen, seeds, tag):
    """joint grid search of several spec keys (full factorial) on the dev seeds. selection: success ↓ · violation ↑ · speed ↓ (efficiency breaks ties). returns (best kwargs, table, episodes)"""
    import itertools
    keys = list(grid); jobs = []; kws = {}
    for combo in itertools.product(*[grid[k] for k in keys]):
        kw = dict(zip(keys, [float(c) for c in combo])); lab = f"tune_{tag}_" + "_".join(f"{k}={v}" for k, v in kw.items()); kws[lab] = kw
        jobs += make_jobs({name: dict(PSPECS[name], **kw)}, [name], scen, seeds, lab)
    d = run_jobs2(jobs, ckpt=CKPT2); d["violation_m"] = violation_of(d)
    tab = d.groupby("label").agg(success=("success", "mean"), violation=("violation_m", "mean"), speed=("mean_speed", "mean"), finished=("finished", "mean")).reset_index()
    tab["mo_cv_success"] = tab.label.map(d[d.scenario == "mo_cv"].groupby("label").success.mean()); tab["mo_cv_finished"] = tab.label.map(d[d.scenario == "mo_cv"].groupby("label").finished.mean())
    for k in keys: tab[k] = tab.label.map(lambda l: kws[l][k])
    tab = tab.sort_values(["success", "violation", "speed"], ascending=[False, True, False]).reset_index(drop=True)
    print(f"--- joint tuning of {keys} for {name} on dev seeds {seeds} over {scen} ---"); print(tab[keys + ["success", "finished", "mo_cv_success", "violation", "speed"]].round(3).to_string(index=False))
    tab.drop(columns="label").to_csv(f"{PO}/tune_{tag}.csv", index=False)
    return {k: float(tab.loc[0, k]) for k in keys}, tab, d

STOP_ON_GATE = True          # set False only if you want to run despite a degenerate scenario set / baseline
TUNE_FILE = f"{PO}/tuned.json"
if os.path.exists(TUNE_FILE): TUNED = json.load(open(TUNE_FILE))
else:
    c_best, d_c = tune_param("Adaptive+TunedInflation", "infl", PC["infl_grid"], SCN2, PC["dev"])
    TUNED = dict(infl=c_best)
    for tg, nm in (("cbf_aware", "Adaptive-MPC + CBF filter"), ("cbf_agnostic", "Adaptive-Track + CBF filter")):
        best_, tab_, d_ = tune_grid(nm, dict(cbf_gamma=PC["cbf_gamma_grid"], cbf_r=PC["cbf_grid"]), SCN2, PC["dev"], tg)
        TUNED[tg] = best_; TUNED[tg + "_dev"] = {k: float(tab_.loc[0, k]) for k in ("success", "finished", "mo_cv_success", "mo_cv_finished", "violation", "speed")}
    json.dump(TUNED, open(TUNE_FILE, "w"), indent=1)
PSPECS["Adaptive+TunedInflation"]["infl"] = TUNED["infl"]; PSPECS["Adaptive-MPC + CBF filter"].update(TUNED["cbf_aware"]); PSPECS["Adaptive-Track + CBF filter"].update(TUNED["cbf_agnostic"])
print("tuned on dev seeds:", TUNED)
# ---- baseline sanity (added after the first quick run, in which the CBF baselines braked to ~9 m/s and never reached x = 100 m: a crawling baseline is not a baseline).
# On the CONTROL scenario mo_cv (dev seeds) a tuned baseline must at least reach the goal in half of the episodes.
CBF_BAD = [k for k in ("cbf_aware", "cbf_agnostic") if TUNED[k + "_dev"]["mo_cv_finished"] < 0.5]
print("baseline sanity (dev, mo_cv):", {k: (round(TUNED[k + "_dev"]["mo_cv_success"], 2), round(TUNED[k + "_dev"]["mo_cv_finished"], 2)) for k in ("cbf_aware", "cbf_agnostic")})
if CBF_BAD and STOP_ON_GATE: raise RuntimeError(f"CBF baseline(s) {CBF_BAD} still do not reach x = 100 m on the control scenario after tuning (gamma_f, r_f). Widen the grid (dev seeds only) or fix the filter; do NOT report such a baseline.")

# ---- difficulty gate (Adaptive-MPC, dev seeds)
dd = run_jobs2(make_jobs(PSPECS, ["Adaptive-MPC (FricID)"], SCN2, PC["dev"], "gate"), ckpt=CKPT2)
gate_tab = dd.groupby("scenario").agg(success=("success", "mean"), collision=("collision", "mean"), road=("road_violation", "mean"), spin=("terminated", "mean"), min_clear=("min_clearance", "mean")).reindex(SCN2)
print("\nDifficulty gate — Adaptive-MPC on DEV seeds:"); print(gate_tab.round(2).to_string())
GATE_MSG = []
if len(PC["dev"]) >= 2:
    if gate_tab.loc["mo_cv", "success"] < 0.75: GATE_MSG.append("mo_cv (control) is not solved by Adaptive-MPC → the world/perception setup is too hard or buggy")
    sh = gate_tab.loc[[s for s in SHIFT2 if s in gate_tab.index], "success"]
    if (sh >= 0.95).all(): GATE_MSG.append("ALL shift scenarios are ≥ 95 % for Adaptive-MPC → no room for any method to win (increase `dv` / noise sigma in SCEN, dev seeds only)")
    if (sh <= 0.05).all(): GATE_MSG.append("ALL shift scenarios are ≤ 5 % for Adaptive-MPC → infeasible; reduce severity")
for m_ in GATE_MSG: print("‼ GATE:", m_)
if GATE_MSG and STOP_ON_GATE: raise RuntimeError("Scenario-difficulty gate fired (see messages above). Adjust the severity on the DEV seeds, delete the pilot_* folder and re-run, or set STOP_ON_GATE = False.")

### P7.2 Main run on the *test* seeds
Resumable (re-run the cell after a disconnect). On Colab mount Google Drive and set `PO` to a Drive path if you want it to survive a runtime reset.

In [ ]:
t0 = time.time()
df2 = run_jobs2(make_jobs(PSPECS, list(PSPECS), SCN2, PC["seeds"], "main"), ckpt=CKPT2)
df2["controller"] = pd.Categorical(df2["controller"], categories=list(PSPECS), ordered=True); df2["violation_m"] = violation_of(df2)
df2 = df2.sort_values(["scenario", "seed", "controller"]).reset_index(drop=True)
if "crashed" in df2 and df2["crashed"].fillna(0).sum() > 0: print("WARNING crashed episodes (counted as failures):", int(df2["crashed"].sum()))
df2.to_csv(f"{PO}/episodes_main.csv", index=False)
print(f"main run: {len(df2)} episodes ({time.time()-t0:.0f}s this session)")

## P8. Results
### P8.1 Tables (Wilson 95 % CIs; success = no collision with the TRUE obstacles, no road departure > 5 cm, no spin-out, AND the vehicle reaches x = 100 m within the episode)

In [ ]:
ORDER = list(PSPECS)
T_shift = success_table(df2[df2.scenario.isin(SHIFT2)], ORDER); T_ctrl = success_table(df2[df2.scenario == "mo_cv"], ORDER)
cols = ["n", "Success %", "CI lo", "CI hi", "Collision %", "Road-dep. %", "Spin-out %", "Safe % (no finish req.)", "Finished %", "Min clear. m", "Speed m/s", "t95 s", "Jerk rms", "Solve ms", "p95 ms", "Deadline miss"]
print("=== POOLED SHIFT SCENARIOS", SHIFT2, "==="); print(T_shift[cols].round(2).to_string())
print("\n=== CONTROL SCENARIO mo_cv ==="); print(T_ctrl[cols].round(2).to_string())
SC_TAB = df2.pivot_table(index="controller", columns="scenario", values="success", aggfunc="mean", observed=False).reindex(ORDER)[SCN2] * 100
print("\n=== success % per scenario ==="); print(SC_TAB.round(1).to_string())
T_shift.round(3).to_csv(f"{PO}/table_pooled_shift.csv"); T_ctrl.round(3).to_csv(f"{PO}/table_control.csv"); SC_TAB.round(2).to_csv(f"{PO}/table_per_scenario.csv")
OBS_CTRL = [c for c in ORDER if PSPECS[c].get("obs_tube")]
cov_tab = df2[df2.controller.isin(OBS_CTRL)].groupby(["controller", "scenario"], observed=True).agg(covo=("covo_overall", "mean"), covo_g1=("covo_g1", "mean"), covo_g2=("covo_g2", "mean"), covo_g3=("covo_g3", "mean"), qo_mid=("qo_mid", "mean"), qo_end=("qo_end", "mean")).round(3)
print("\n=== obstacle-margin coverage (target 0.90) and mean margin at j=10 / 25 [m] ==="); print(cov_tab.to_string()); cov_tab.to_csv(f"{PO}/table_obstacle_coverage.csv")

### P8.2 Paired tests (by scenario × seed), positive difference = the first-named controller is better

In [ ]:
PW = {}
for ref in (PRIMARY, SECONDARY):
    PW[ref] = paired_vs(df2, ref, [c for c in ORDER if c != ref], SHIFT2); print(f"\n=== {ref} vs others | pooled shift scenarios ==="); print(PW[ref].round(4).to_string()); PW[ref].round(5).to_csv(f"{PO}/paired_{ref.split(' ')[0]}.csv")
PW_ctrl = paired_vs(df2, PRIMARY, B4, ["mo_cv"]); print("\n=== control scenario mo_cv: PRIMARY vs the 4 strong baselines ==="); print(PW_ctrl.round(4).to_string())
PW_scn = {s: paired_vs(df2, PRIMARY, B4, [s]) for s in SHIFT2}
print("\n=== per-scenario success difference PRIMARY − baseline [pp] ==="); print(pd.DataFrame({s: t["diff_pp"] for s, t in PW_scn.items() if len(t)}).round(1).to_string())

### P8.3 Pre-specified hypotheses and the automatic decision
* **HA (decisive)** — `CA-NMPC-Obs` beats **each** of the 4 strong baselines by ≥ 10 pp on the pooled shift scenarios with Holm-adjusted exact-McNemar p < 0.05.  *(the same test is reported for `Adaptive+ObsACI`)*
* **HB** — the NN residual + ego tube add value on top of the obstacle margin: `CA-NMPC-Obs` > `Adaptive+ObsACI` (Holm-free single comparison, p < 0.05).
* **HC** — online adaptation matters: `Adaptive+ObsACI` > `Adaptive+ObsStaticCP` (p < 0.05).
* **HD** — no harm in the control scenario: `CA-NMPC-Obs` success on `mo_cv` ≥ best-baseline − 10 pp.
* **HE** — obstacle-margin coverage of `Adaptive+ObsACI` within ±0.05 of 0.90 in ≥ 80 % of (shift scenario × horizon group) cells.

In [ ]:
def ha(ref):
    t = PW[ref].loc[[b for b in B4 if b in PW[ref].index]]
    ok = (t.diff_pp >= 10) & (t.p_McNemar_holm < 0.05)
    return bool(ok.all()) and len(t) == len(B4), t
HA_P, tP = ha(PRIMARY); HA_S, tS = ha(SECONDARY)
def one(a, b, scen):
    t = paired_vs(df2, a, [b], scen); r = t.iloc[0]; return r.diff_pp, r.p_McNemar
dB, pB = one(PRIMARY, SECONDARY, SHIFT2); dC, pC = one(SECONDARY, "Adaptive+ObsStaticCP", SHIFT2)
ctrl_best = max(T_ctrl.loc[b, "Success %"] for b in B4); HD_ok = bool(T_ctrl.loc[PRIMARY, "Success %"] >= ctrl_best - 10)
cc = cov_tab.loc[SECONDARY] if SECONDARY in cov_tab.index.get_level_values(0) else None
cells = [abs(cc.loc[s, g] - 0.9) <= 0.05 for s in SHIFT2 if cc is not None and s in cc.index for g in ("covo_g1", "covo_g2", "covo_g3")] if cc is not None else []
HE_frac = float(np.mean(cells)) if cells else np.nan
HYP2 = pd.DataFrame([
    dict(h="HA", claim=f"{PRIMARY} beats ALL 4 strong baselines (≥ +10 pp, Holm p < 0.05)", verdict="SUPPORTED" if HA_P else "NOT SUPPORTED", detail="; ".join(f"{b}: {r.diff_pp:+.1f} pp (p_holm {r.p_McNemar_holm:.3f})" for b, r in tP.iterrows())),
    dict(h="HA'", claim=f"{SECONDARY} beats ALL 4 strong baselines (same rule)", verdict="SUPPORTED" if HA_S else "NOT SUPPORTED", detail="; ".join(f"{b}: {r.diff_pp:+.1f} pp (p_holm {r.p_McNemar_holm:.3f})" for b, r in tS.iterrows())),
    dict(h="HB", claim="NN residual + ego tube add value on top of the obstacle margin", verdict="SUPPORTED" if (dB > 0 and pB < 0.05) else ("CONTRADICTED" if (dB < 0 and pB < 0.05) else "inconclusive"), detail=f"{PRIMARY} − {SECONDARY}: {dB:+.1f} pp (p {pB:.3f})"),
    dict(h="HC", claim="online adaptation (ACI) beats the static split-CP obstacle margin", verdict="SUPPORTED" if (dC > 0 and pC < 0.05) else ("CONTRADICTED" if (dC < 0 and pC < 0.05) else "inconclusive"), detail=f"{SECONDARY} − ObsStaticCP: {dC:+.1f} pp (p {pC:.3f})"),
    dict(h="HD", claim="no harm in the control scenario mo_cv", verdict="SUPPORTED" if HD_ok else "NOT SUPPORTED", detail=f"{PRIMARY}: {T_ctrl.loc[PRIMARY, 'Success %']:.1f} % vs best baseline {ctrl_best:.1f} %"),
    dict(h="HE", claim="obstacle-margin coverage of Adaptive+ObsACI within ±0.05 of 0.90", verdict="SUPPORTED" if (cells and HE_frac >= 0.8) else "NOT SUPPORTED", detail=f"{100*HE_frac:.0f} % of cells"),
])
pd.set_option("display.max_colwidth", 200); print(HYP2.to_string(index=False)); HYP2.to_csv(f"{PO}/hypotheses.csv", index=False)
if HA_P: DECISION = "PATH A is viable: the proposed controller beats all four strong baselines under the pre-specified rule. Next: 30 seeds on the second plant, GP-MPC / safe-RL baselines, a novelty check against egocentric-CP and SODA-MPC, then theory."
elif HA_S: DECISION = "PATH A' — the obstacle-side conformal margin (without NN, without ego tube) beats all four baselines, but the proposed full controller does not: the contribution would be the obstacle-side delayed-feedback ACI margin, and the NN/ego-tube part cannot be claimed. Novelty vs egocentric-CP / SODA-MPC must be checked before writing."
else: DECISION = "PATH B — the pre-specified rule for a method paper is NOT met. Report this as an evaluation/benchmark paper (joint-coverage gap, delayed-feedback ACI, gate, honest negative results) rather than adding scenarios until something wins."
print("\nDECISION (automatic):", DECISION)

### P8.4 Figures

In [ ]:
def fig_heat():
    M = SC_TAB.values; fig, ax = plt.subplots(figsize=(1.6 + 1.1 * len(SCN2), 0.45 * len(ORDER) + 1.3))
    im = ax.imshow(M, vmin=0, vmax=100, cmap="Blues", aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]): ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=8, color="white" if M[i, j] > 60 else "black")
    ax.set_xticks(range(len(SCN2))); ax.set_xticklabels(SCN2, rotation=30, ha="right"); ax.set_yticks(range(len(ORDER))); ax.set_yticklabels(ORDER); ax.grid(False)
    ax.set_title("success % (test seeds)"); fig.colorbar(im, ax=ax, fraction=0.03); fig.savefig(f"{PO}/fig_success_heatmap.png"); plt.show()
fig_heat()

def fig_forest():
    fig, axs = plt.subplots(1, 2, figsize=(14, 0.4 * len(ORDER) + 1.4))
    for ax, ref in zip(axs, (PRIMARY, SECONDARY)):
        t = PW[ref]; yy = np.arange(len(t))[::-1]
        for i, (name, r) in enumerate(t.iterrows()):
            sig = r.p_McNemar_holm < 0.05; col = PCOL.get(name, "k")
            ax.plot([r.ci_lo_pp, r.ci_hi_pp], [yy[i]] * 2, color=col, lw=1.6); ax.plot(r.diff_pp, yy[i], "o", ms=6 if sig else 5, mfc=col if sig else "white", mec=col)
        ax.axvline(0, color="k", lw=0.8); ax.axvline(10, color="#999", lw=0.8, ls=":"); ax.set_yticks(yy); ax.set_yticklabels(list(t.index)); ax.set_xlabel(f"success difference [pp] (+ = {ref.split(' (')[0]} better)"); ax.set_title(ref)
    fig.suptitle("paired success differences, pooled shift scenarios (filled = Holm p<0.05; dotted = the +10 pp rule)", fontsize=9); fig.savefig(f"{PO}/fig_forest.png"); plt.show()
fig_forest()

def fig_trace(scn="mo_swerve", seed=None, names=("Adaptive-MPC (FricID)", "Adaptive-Track + CBF filter", "Adaptive+ObsACI", "CA-NMPC-Obs (ours)")):
    if scn not in SCN2: return
    seed = seed or PC["seeds"][0]; names = [n for n in names if n in PSPECS]
    runs = {n: simulate2(n, PSPECS[n], scn, seed, keep=True) for n in names}
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.2)); ex0 = runs[names[0]][1]; OB = ex0["OBt"]
    axs[0].plot(OB[:, 0, 0], OB[:, 0, 1], "k--", lw=1.2, label="obstacle 1 (true path)"); axs[0].plot(ex0["OBm"][:, 0, 0], ex0["OBm"][:, 0, 1], ".", color="#bbb", ms=2, label="obstacle 1 (measured)")
    axs[0].plot(OB[:, 1, 0], OB[:, 1, 1], "k*", ms=8, label="obstacle 2 (static)")
    for n in names:
        m_, ex_ = runs[n]; axs[0].plot(ex_["Str"][:, 0], ex_["Str"][:, 1], color=PCOL.get(n, "k"), lw=1.8, label=f"{n} ({'ok' if m_['success'] else 'FAIL'}; clear {m_['min_clearance']:.2f} m)")
    axs[0].axhline(Y_LO, color="k", lw=0.8); axs[0].axhline(Y_HI, color="k", lw=0.8); axs[0].set_xlabel("x [m]"); axs[0].set_ylabel("y [m]"); axs[0].set_title(f"{scn} seed {seed}"); axs[0].legend(fontsize=6.5)
    for n in names:
        m_, ex_ = runs[n]; ql = ex_["qolog"]
        if np.any(ql > 0): tt = np.arange(len(ql)) * DT; axs[1].plot(tt, ql[:, 9], color=PCOL.get(n, "k"), lw=1.6, label=f"{n}: j=10"); axs[1].plot(tt, ql[:, -1], color=PCOL.get(n, "k"), lw=1.0, ls="--", label=f"{n}: j=25")
    axs[1].set_xlabel("t [s]"); axs[1].set_ylabel("obstacle-motion margin [m]"); axs[1].set_title("obstacle-side margin growth after the manoeuvre"); axs[1].legend(fontsize=6.5)
    for n in names:
        m_, ex_ = runs[n]; P = ex_["Str"][:, :2]; T = len(P); d = np.linalg.norm(OB[:T, 0, :] - P, axis=1) - R_O - R_V; axs[2].plot(np.arange(T) * DT, d, color=PCOL.get(n, "k"), lw=1.6, label=n)
    axs[2].axhline(0, color="r", lw=0.8); axs[2].set_xlabel("t [s]"); axs[2].set_ylabel("clearance to obstacle 1 [m]"); axs[2].set_title("clearance (negative = collision)"); axs[2].legend(fontsize=6.5)
    fig.savefig(f"{PO}/fig_trace_{scn}.png"); plt.show()
fig_trace("mo_swerve"); fig_trace("mo_swerve_noisy")

def fig_tradeoff():
    fig, ax = plt.subplots(figsize=(6.5, 4.4))
    for c in ORDER:
        x = df2[(df2.controller == c) & df2.scenario.isin(SHIFT2)]; k, n = int(x.success.sum()), len(x); lo, hi = wilson(k, n)
        ax.errorbar(x.mean_speed.mean(), 100 * k / n, yerr=[[100 * (k / n - lo)], [100 * (hi - k / n)]], fmt="o", color=PCOL.get(c, "k"), capsize=2, ms=6, label=c)
    ax.set_xlabel("mean speed [m/s] (efficiency)"); ax.set_ylabel("success % (pooled shift scenarios)"); ax.legend(fontsize=6.5, loc="upper left"); ax.set_title("safety vs efficiency"); fig.savefig(f"{PO}/fig_tradeoff.png"); plt.show()
fig_tradeoff()

## P9. Export: one report to send back + all tables/figures/raw episodes
`PILOT_REPORT.md` contains everything needed to judge the pilot (configuration, regression test, Step 1 diagnosis and fix, tuned baselines, difficulty gate, pooled / per-scenario results, paired tests, hypotheses, automatic decision).
**Send me `PILOT_REPORT.md` (inside the zip) and, if you want, `episodes_main.csv`.**

In [ ]:
import shutil, platform, glob, hashlib
PROTO2 = dict(pilot=PILOT, plant=PLANT, delay=DELAY, test_seeds=PC["seeds"], diag_seeds=PC["diag"], dev_seeds=PC["dev"], scenarios=SCN2, controllers=list(PSPECS), tuned=TUNED, step1_fix=FIX,
              alpha=ALPHA, eta=ETA, gamma=GAMMA, obs_qcap=OBS_QCAP, N=N, DT=DT, T_sim=CFG["T_sim"], scen={k: SCEN[k] for k in SCN2 + ["tire_stiff"]}, regression_ok=bool(REG_OK))
PHASH = hashlib.sha256(json.dumps(PROTO2, sort_keys=True, default=str).encode()).hexdigest()[:12]
json.dump(dict(PROTO2, hash=PHASH), open(f"{PO}/protocol.json", "w"), indent=1, default=str)
fence = lambda df_, nd=2: "```\n" + df_.round(nd).to_string() + "\n```\n"
R = [f"# Pilot report — {PILOT} / plant {PLANT} / delay {DELAY}", f"protocol hash `{PHASH}` · python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · casadi {ca.__version__}",
     f"test seeds: {len(PC['seeds'])} ({PC['seeds'][0]}…{PC['seeds'][-1]}) · diag seeds: {len(PC['diag'])} · dev seeds: {PC['dev']} · scenarios: {SCN2}",
     f"\n**Regression test simulate2 vs simulate:** {'OK' if REG_OK else 'MISMATCH — do not trust the results'}",
     "\n## Step 1 — tire_stiff diagnosis (diag seeds)", fence(T1[["n", "Success %", "CI lo", "CI hi", "Collision %", "Road-dep. %", "Spin-out %", "q_mid", "q_end", "slack_margin", "tau", "xi_end"]]),
     "```\n" + "\n".join(diagnose_step1(df1)) + "\n```", f"Fix chosen on the diag seeds and frozen for Step 2: `{FIX}`",
     "\n## Tuned baselines (dev seeds)", f"`{TUNED}`", "\n## Difficulty gate (Adaptive-MPC, dev seeds)", fence(gate_tab), ("**GATE MESSAGES:** " + "; ".join(GATE_MSG)) if GATE_MSG else "gate: passed",
     "\n## Step 2 — pooled shift scenarios (test seeds)", fence(T_shift[cols]), "\n### control scenario mo_cv", fence(T_ctrl[cols]), "\n### success % per scenario", fence(SC_TAB, 1),
     "\n### obstacle-margin coverage (target 0.90) and margins", fence(cov_tab, 3), f"\n### paired tests: {PRIMARY} vs others (pooled shift)", fence(PW[PRIMARY], 3), f"\n### paired tests: {SECONDARY} vs others (pooled shift)", fence(PW[SECONDARY], 3),
     "\n### per-scenario difference PRIMARY − baseline [pp]", fence(pd.DataFrame({s: t["diff_pp"] for s, t in PW_scn.items() if len(t)}), 1),
     "\n## Hypotheses (pre-specified)", "```\n" + HYP2.to_string(index=False) + "\n```", f"\n## DECISION (automatic)\n{DECISION}",
     "\n## Known limitations of this pilot\n- simulation only; the second obstacle is static; obstacle motion changes are abrupt but piece-wise constant (swerve / stop), not adversarial\n- obstacle scores compare measurements with measurements: a constant bias or latency is invisible to them (mo_bias is the negative control, not run by default)\n"
     "- the CBF filter is a predictive (look-ahead) filter solved with IPOPT, not a learned CBF; GP-MPC and safe-RL baselines are NOT included\n- no recursive-feasibility result; the delayed-feedback ACI bound (Prop. 3 of the main notebook) does not cover the grouped/windowed implementation\n- scenario severity in SCEN was fixed by hand and only checked by the difficulty gate; IPOPT solve times (~0.05-0.25 s) exceed the 0.1 s period for the NN controllers (deadline_miss column)"]
open(f"{PO}/PILOT_REPORT.md", "w").write("\n".join(R))
zpath = shutil.make_archive(f"pilot_results_{PLANT}_{PILOT}", "zip", PO)
print("saved", zpath, "|", len(glob.glob(f"{PO}/*")), "files in", PO, "\n→ send PILOT_REPORT.md (and episodes_main.csv) for evaluation")
try:
    from google.colab import files; files.download(zpath)
except Exception: pass

## What to do with the result
* **Send `PILOT_REPORT.md`** (inside `pilot_results_*.zip`). It contains the Step 1 verdict, the frozen fix, the tuned baselines, all result tables and the automatic decision.
* **How to read it.** Step 1 tells whether the tire-stiffness failure was a *design flaw* (e.g. margins on both road edges), a *bug*, or a *genuine weakness*. Step 2 answers one question only: does an online-calibrated margin beat the strong baselines on moving-obstacle shifts by a pre-specified margin?
* **If the decision is path B**, the right move is *not* to add scenarios until something wins, but to write the evaluation paper (joint-coverage gap, delayed-feedback ACI, residual gate, honest negative results).
* **Not covered:** GP-MPC, safe RL, learned CBFs, real data, CARLA/BeamNG, recursive feasibility, a second moving obstacle, adversarial obstacles.